In [ ]:
%load_ext autoreload
%autoreload 2
# %flow mode reactive

from datetime import datetime
import sys
import os
from pathlib import Path
from typing import Any, Tuple, List, Dict
from dotmap import DotMap
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import plotly
import plotly.express as px
import plotly.graph_objs as go
import plotly.io as pio
from plotly.subplots import make_subplots
import statsmodels.api as sm
from scipy.stats import ttest_rel
from tqdm.notebook import tqdm

import datajoint as dj
from aeon.dj_pipeline.analysis.block_analysis import *
from aeon.dj_pipeline import acquisition, streams
from swc.aeon.io import api as aeon_api
from swc.aeon.io import reader as aeon_reader
from aeon.schema.schemas import social02, exp02
from aeon.analysis.movies import gridframes
from aeon.io.video import frames

cwd = os.getcwd()
project_path = os.path.join(cwd, "ProjectAeon", "aeon_scratchpad", "aeon_analysis", "aeon_methods_paper")
sys.path.append(project_path)
from data_io_utils import save_all_experiment_data, load_data_from_parquet

# Definitions

In [ ]:
data_dir = Path("/ceph/aeon/aeon/code/scratchpad/methods_paper_data")
save_dir = Path("/ceph/aeon/aeon/code/scratchpad/anaya/methods_paper")
os.makedirs(data_dir, exist_ok=True)
cm2px = 5.2  # 1 cm = 5.2 px roughly in aeon arenas
light_off, light_on = 7, 20  # 7am to 7pm
fps = 50

In [ ]:
experiments = [
    {"name": "social0.2-aeon3", "presocial_start": '2024-01-31 11:00:00', "presocial_end": '2024-02-08 15:00:00', "social_start": '2024-02-09 16:00:00', "social_end": '2024-02-23 13:00:00', "postsocial_start": '2024-02-25 17:00:00', "postsocial_end": '2024-03-02 14:00:00'},
    {"name": "social0.2-aeon4", "presocial_start": '2024-01-31 11:00:00', "presocial_end": '2024-02-08 15:00:00', "social_start": '2024-02-09 17:00:00', "social_end": '2024-02-23 12:00:00', "postsocial_start": '2024-02-25 18:00:00', "postsocial_end": '2024-03-02 13:00:00'},
    {"name": "social0.3-aeon3", "presocial_start": '2024-06-08 19:00:00', "presocial_end": '2024-06-17 13:00:00', "social_start": '2024-06-25 11:00:00', "social_end": '2024-07-06 13:00:00', "postsocial_start": '2024-07-07 16:00:00', "postsocial_end": '2024-07-14 14:00:00'},
    {"name": "social0.3-aeon4", "presocial_start": '2024-06-08 19:00:00', "presocial_end": '2024-06-17 14:00:00', "social_start": '2024-06-19 12:00:00', "social_end": '2024-07-03 14:00:00', "postsocial_start": '2024-07-04 11:00:00', "postsocial_end": '2024-07-13 12:00:00'},
    {"name": "social0.4-aeon3", "presocial_start": '2024-08-16 17:00:00', "presocial_end": '2024-08-24 10:00:00', "social_start": '2024-08-28 11:00:00', "social_end": '2024-09-09 13:00:00', "postsocial_start": '2024-09-09 18:00:00', "postsocial_end": '2024-09-22 16:00:00'},
    {"name": "social0.4-aeon4", "presocial_start": '2024-08-16 15:00:00', "presocial_end": '2024-08-24 10:00:00', "social_start": '2024-08-28 10:00:00', "social_end": '2024-09-09 01:00:00', "postsocial_start": '2024-09-09 15:00:00', "postsocial_end": '2024-09-22 16:00:00'}
]
experiment = experiments[0]

In [ ]:
def load_experiment_data(data_dir, experiment=None, periods=None, 
                        data_types=['rfid', 'position'], trim_days=None):
    """
    Load all data types for specified periods of an experiment.
    
    Parameters:
    - experiment: experiment dict with period start/end times
    - periods: list of periods to load
    - data_types: list of data types to load
    - data_dir: directory containing data files
    - trim_days: Optional number of days to trim from start (None = no trim)
    
    Returns:
    - Dictionary containing dataframes for each period/data type combination
    """
    
    result = {}

    if periods is None:
        periods = [None]
    
    for period in periods:
        for data_type in data_types:
            print(f"Loading {period} {data_type} data...")
            
            # Load data
            if experiment is not None:
                experiment_name = experiment["name"]
            else:
                experiment_name = None
            df = load_data_from_parquet(
                experiment_name=experiment_name,
                period=period,
                data_type=data_type,
                data_dir=data_dir,
                set_time_index=(data_type == 'position')
            )
            
            # Trim if requested
            if trim_days is not None and len(df) > 0:
                if data_type == 'rfid':
                    start_time = df['chunk_start'].min()
                    end_time = start_time + pd.Timedelta(days=trim_days)
                    df = df[df['chunk_start'] < end_time]
                if data_type == 'foraging':
                    start_time = df['start'].min()
                    end_time = start_time + pd.Timedelta(days=trim_days)
                    df = df[df['start'] < end_time]
                if data_type == 'position':
                    start_time = df.index.min()
                    end_time = start_time + pd.Timedelta(days=trim_days)
                    df = df.loc[df.index < end_time]
                
                print(f"  Trimmed to {trim_days} days: {len(df)} records")
            
            # Store in result
            key = f"{period}_{data_type}"
            result[key] = df
            
            # For position data, handle duplicates
            if data_type == 'position' and len(df) > 0:
                original_len = len(df)
                df = df.reset_index()
                df = df.drop_duplicates(subset=['time', 'identity_name'])
                df = df.set_index('time')
                result[key] = df
                if len(df) < original_len:
                    print(f"  Removed duplicates: {original_len} -> {len(df)}")
    
    return result

In [ ]:
def clean_swaps(df: pd.DataFrame) -> pd.DataFrame:
    """
    Fast swap‐correction that returns the original df's columns,
    but with x,y replaced by the cleaned tracks. Includes identity correction
    based on SLEAP's majority vote *inside* the main loop, so we only loop once over T.

    Steps:
      0) sort & get identities
      1) reset_index so we can merge back later
      2) pivot x,y into 2×T arrays
      3) prepare cleaned arrays
      4) find first fully‐observed column and initialize
      5) loop over t=first_i+1..T-1:
            • swap‐correction logic (same as before)
            • **immediately** update track_votes whenever x_raw[:,t] & x_clean[:,t] are both finite
      6) decide final swap based on accumulated votes
      7) rebuild cleaned‐coord DataFrame & merge back
    """

    # 0) sort & get identities
    df = df.sort_index()
    ids = df['identity_name'].unique()
    assert len(ids) == 2, "Need exactly two identities"

    # 1) reset_index so we can merge back later
    df2 = df.reset_index()
    time_col = df2.columns[0]  # timestamp column name

    # 2) pivot x,y into 2×T arrays
    wide = df2.pivot(index=time_col, columns='identity_name', values=['x','y'])
    times = wide.index.values
    T = len(times)
    x_raw = np.vstack([wide['x'][ids[0]].values,
                       wide['x'][ids[1]].values])
    y_raw = np.vstack([wide['y'][ids[0]].values,
                       wide['y'][ids[1]].values])

    # 3) prepare cleaned arrays
    x_clean = np.full_like(x_raw, np.nan)
    y_clean = np.full_like(y_raw, np.nan)

    # 4) find first fully‐observed column
    valid = np.isfinite(x_raw).all(axis=0)
    first_i = np.argmax(valid)
    last_x = x_raw[:, first_i].copy()
    last_y = y_raw[:, first_i].copy()
    x_clean[:, first_i] = last_x
    y_clean[:, first_i] = last_y

    # Initialize vote‐matrix: [cleaned_track, original_identity]
    track_votes = np.zeros((2, 2), dtype=np.int64)

    # If the very first frame is fully observed, count those votes now:
    if valid[first_i]:
        # At t=first_i, x_clean[:,first_i] == x_raw[:,first_i], so track 0 ← orig 0, track 1 ← orig 1
        track_votes[0, 0] += 1
        track_votes[1, 1] += 1

    # 5) loop and swap‐correct *and* vote in one pass
    for t in tqdm(range(first_i + 1, T), desc="Cleaning frames"):
        # 5a) if not all raw‐points are finite, just carry‐forward
        if not np.isfinite(x_raw[:, t]).all():
            for k in (0, 1):
                if np.isfinite(x_raw[k, t]):
                    x_clean[k, t] = last_x[k]
                    y_clean[k, t] = last_y[k]
            # No "vote" in incomplete frames
            continue

        # 5b) if the two mice are very close, keep original order
        inter_mouse_dist = np.hypot(x_raw[0, t] - x_raw[1, t],
                                    y_raw[0, t] - y_raw[1, t])
        if inter_mouse_dist < 100:
            x_clean[:, t] = x_raw[:, t]
            y_clean[:, t] = y_raw[:, t]
            # immediate vote: clean track 0 came from raw‐0, clean 1 from raw‐1
            track_votes[0, 0] += 1
            track_votes[1, 1] += 1
            last_x = x_raw[:, t].copy()
            last_y = y_raw[:, t].copy()
            continue

        # 5c) compute distances to previous frame to decide swap
        dx = x_raw[:, t][:, None] - last_x[None, :]
        dy = y_raw[:, t][:, None] - last_y[None, :]
        dist = np.hypot(dx, dy)
        d_same = dist[0, 0] + dist[1, 1]
        d_swap = dist[0, 1] + dist[1, 0]

        # 5d) if even the best assignment is too big, carry‐forward
        if min(d_same, d_swap) > 90:
            x_clean[:, t] = last_x
            y_clean[:, t] = last_y
            # No vote, because x_clean did not actually come from x_raw this frame
            continue

        # 5e) pick same vs swapped assignment
        if d_same <= d_swap:
            x_clean[:, t] = x_raw[:, t]
            y_clean[:, t] = y_raw[:, t]
            # vote: clean₀ came from raw₀, clean₁ from raw₁
            track_votes[0, 0] += 1
            track_votes[1, 1] += 1
        else:
            x_clean[:, t] = x_raw[::-1, t]
            y_clean[:, t] = y_raw[::-1, t]
            # vote: clean₀ came from raw₁, clean₁ from raw₀
            track_votes[0, 1] += 1
            track_votes[1, 0] += 1

        # 5f) update last_x/last_y
        last_x = x_clean[:, t].copy()
        last_y = y_clean[:, t].copy()

    # 6) Final identity swap based on majority vote
    need_swap = track_votes[0, 1] > track_votes[0, 0]
    if need_swap:
        print(f"Swapping final tracks based on SLEAP majority vote")
        print(f"Track votes:\n{track_votes}")
        x_clean = x_clean[::-1, :]
        y_clean = y_clean[::-1, :]

    # 7) build cleaned coord table, naming columns x,y
    cleaned = pd.DataFrame({
        time_col: np.repeat(times, 2),
        'identity_name': np.tile(ids, T),
        'x': x_clean.ravel(order='F'),
        'y': y_clean.ravel(order='F'),
    })

    # 8) drop the old x,y and merge back everything else
    df2_noxy = df2.drop(columns=['x', 'y'])
    result = (
        df2_noxy
        .merge(cleaned, on=[time_col, 'identity_name'], how='right')
        .set_index(time_col)
        .sort_index()
    )

    return result


# SLEAP and RFID plots

## Load data

In [ ]:
# Load all periods for experiment
data = load_experiment_data(
    experiment=experiment,
    data_dir=data_dir,
    periods=['presocial', 'social', 'postsocial'],
    data_types=['rfid', 'position'],
    # trim_days=1  # Optional: trim
)

# Access data
presocial_rfid_df = data['presocial_rfid']
presocial_position_df = data['presocial_position']
social_rfid_df = data['social_rfid']
social_position_df = data['social_position']
postsocial_rfid_df = data['postsocial_rfid']
postsocial_position_df = data['postsocial_position']

In [ ]:
social_position_df = clean_swaps(social_position_df)  # Clean swaps in social position data

In [ ]:
# # Optional: restrict time to chunk used for sleap training
# social_position_df = social_position_df.loc['2024-02-10 11:00:00':'2024-02-10 11:59:59']
# social_rfid_df = social_rfid_df.loc[social_rfid_df['chunk_start'] >= '2024-02-10 11:00:00']
# social_rfid_df = social_rfid_df.loc[social_rfid_df['chunk_start'] < '2024-02-10 12:00:00']

In [ ]:
# RFID patches 1 and 2 are swapped in the presocial period of the social0.2-aeon3 experiment
if experiment["name"] == "social0.2-aeon3":
    # First, create a temporary placeholder to avoid overwriting during the swap
    presocial_rfid_df['rfid_reader_name'] = presocial_rfid_df['rfid_reader_name'].replace({
        'Patch1Rfid': 'TEMP_PLACEHOLDER',
        'Patch2Rfid': 'Patch1Rfid'
    })
    # Now replace the placeholder with Patch2Rfid
    presocial_rfid_df['rfid_reader_name'] = presocial_rfid_df['rfid_reader_name'].replace({
        'TEMP_PLACEHOLDER': 'Patch2Rfid'
    })
    # Verify the swap worked
    print(presocial_rfid_df["rfid_reader_name"].unique())

In [ ]:
acquisition_computer = experiment["name"].split("-")[1].upper()
social_name = experiment["name"].split("-")[0]
metadata_root = Path(f"/ceph/aeon/aeon/data/raw/{acquisition_computer}/{social_name}")
metadata_reader = social02.Metadata
metadata = aeon_api.load(metadata_root, metadata_reader)['metadata'].iloc[0]
rfid_devices_loc = DotMap({key: metadata.Devices[key].Location for key in metadata.Devices.keys() if 'rfid' in key.lower()})
rfid_devices_loc

## Match RFID and Pose rows based on time and position

In [ ]:
def match_rfid_to_pose(rfid_df, pose_df, rfid_devices_loc, cm2px, tolerance_ms=10):
    """
    Match RFID timestamps to nearest pose timestamps within tolerance.
    Returns dataframe with matched data, including unmatched RFID rows.
    """
    # Explode RFID data to have one row per timestamp/identity
    rfid_exploded = (
    rfid_df[['rfid_reader_name', 'timestamps', 'rfid']]
    .explode(['timestamps', 'rfid'])
    .rename(columns={'timestamps': 'rfid_time', 'rfid': 'rfid_identity'})
    )
    rfid_exploded['rfid_time'] = pd.to_datetime(rfid_exploded['rfid_time'])
    rfid_exploded = rfid_exploded.drop_duplicates(subset=['rfid_time'], keep='first')
    rfid_valid = rfid_exploded.dropna(subset=['rfid_time']).copy()
    rfid_valid['row_id'] = range(len(rfid_valid))
    print(rfid_valid.shape)
    
    # Get RFID reader coordinates
    reader_coords = {}
    for name, loc in rfid_devices_loc.items():
        if (name.startswith('_') or 
            name.startswith('*') or 
            callable(getattr(rfid_devices_loc, name, None)) or
            name in ['to_pandas', 'to_dict', 'toDict', 'empty', 'copy']):
            continue
        if hasattr(loc, 'X') and hasattr(loc, 'Y'):
            reader_coords[name] = (float(loc.X), float(loc.Y))
    
    # Add reader coordinates to rfid_valid
    rfid_valid['rfid_x'] = rfid_valid['rfid_reader_name'].map(lambda x: reader_coords.get(x, (None, None))[0])
    rfid_valid['rfid_y'] = rfid_valid['rfid_reader_name'].map(lambda x: reader_coords.get(x, (None, None))[1])
    
    # Prepare pose data for efficient merging
    pose_subset = (
        pose_df[['identity_name', 'identity_likelihood', 'x', 'y', 'likelihood']]
        .reset_index()
        .rename(columns={
            'identity_name': 'pose_identity', 
            'time': 'pose_time', 
            'x': 'pose_x', 
            'y': 'pose_y',
            'identity_likelihood': 'pose_identity_likelihood',
            'likelihood': 'pose_likelihood'
        })
    )
    
    # Merge RFID with all possible pose identities to find all candidates
    unique_identities = pose_subset['pose_identity'].unique()
    rfid_expanded = pd.concat([
        rfid_valid.assign(pose_identity=identity) 
        for identity in unique_identities
    ])
    
    # Use merge_asof to find nearest timestamps within tolerance
    merged = pd.merge_asof(
        rfid_expanded.sort_values('rfid_time'),
        pose_subset.sort_values('pose_time'),
        left_on='rfid_time',
        right_on='pose_time',
        by='pose_identity',
        direction='nearest',
        tolerance=pd.Timedelta(f'{tolerance_ms}ms')
    )
    
    # Calculate spatial distances between reader and animal positions in cm
    merged['rfid_pose_distance'] = np.sqrt(
        (merged['pose_x'] - merged['rfid_x'])**2 + 
        (merged['pose_y'] - merged['rfid_y'])**2
    ) / cm2px
    
    # Keep only the closest match for each RFID detection (by distance)
    idx_closest = merged.groupby('row_id')['rfid_pose_distance'].idxmin()
    valid_idx = idx_closest.dropna()
    
    # Start with all rfid_valid rows
    result = rfid_valid.set_index('row_id')

    # Create empty dataframe with correct structure from merged
    empty_template = merged[['pose_time', 'pose_identity', 'pose_identity_likelihood', 
                            'pose_x', 'pose_y', 'pose_likelihood', 'rfid_pose_distance']].iloc[:0]
    result = result.join(empty_template)
    
    # Update with matched data where available
    if len(valid_idx) > 0:
        matched_data = merged.loc[valid_idx].set_index('row_id')
        result.update(matched_data)
    
    result = result.reset_index()
    
    # Reorder columns
    column_order = [
        'rfid_time', 'rfid_reader_name', 'rfid_identity', 'rfid_x', 'rfid_y',
        'pose_time', 'pose_identity', 'pose_identity_likelihood', 'pose_x', 'pose_y', 
        'pose_likelihood', 'rfid_pose_distance'
    ]
    
    # Select columns that exist in the result
    existing_columns = [col for col in column_order if col in result.columns]
    final_result = result[existing_columns].reset_index(drop=True)
    
    return final_result

In [ ]:
pre_post_social_rfid_df = pd.concat([presocial_rfid_df, postsocial_rfid_df], ignore_index=True)
pre_post_social_position_df = pd.concat([presocial_position_df, postsocial_position_df], ignore_index=False)
pre_post_social_matched_df = match_rfid_to_pose(pre_post_social_rfid_df, pre_post_social_position_df, rfid_devices_loc, tolerance_ms=10, cm2px=cm2px)
print(pre_post_social_matched_df["rfid_time"].isna().sum()) # should be 0
display(pre_post_social_matched_df)
social_matched_df = match_rfid_to_pose(social_rfid_df, social_position_df, rfid_devices_loc, tolerance_ms=10, cm2px=cm2px)
display(social_matched_df)

### 1. RFID readers' range, latency and accuracy

In [ ]:
# Calculate RFID reader ranges
quantiles = [0.95, 0.99, 1.0]
pre_post_social_reader_ranges = (
    pre_post_social_matched_df
    .dropna(subset=['rfid_pose_distance'])
    .groupby('rfid_reader_name')['rfid_pose_distance']
    .agg(['median', 'mean', ('p95', lambda x: x.quantile(0.95)), 
          ('p99', lambda x: x.quantile(0.99)), 'max'])
    .round(1)
)

print("RFID Reader Ranges (cm):")
print(pre_post_social_reader_ranges)
mean_range = pre_post_social_reader_ranges['mean'].mean()
print(f"Mean range: {mean_range:.1f} cm")

In [ ]:
# # Optional debugging
# import swc.aeon.io.reader
# import swc.aeon.io.api
# from aeon.schema.schemas import exp02
# from aeon.analysis.movies import gridframes
# from aeon.io.video import frames
# from aeon.dj_pipeline.analysis.block_analysis import *
# import plotly.express as px
# # Plot the data on the corresponding video frame
# idx = 1
# time = pre_post_social_matched_df.iloc[idx]['pose_time']
# df_to_plot = pre_post_social_matched_df.iloc[idx:idx+1]
# display(df_to_plot)
# root = Path(f"/ceph/aeon/aeon/data/raw/{acquisition_computer}/{social_name}")
# vid_data = swc.aeon.io.api.load(root, exp02.CameraTop.Video, start=time, end=time+pd.Timedelta(seconds=1/fps/2))
# vid_data = vid_data[time:time] # idk why this is necessary by for some reason the video data is loading more than just the ts between time and time+1/fps/2
# fig = px.imshow(gridframes(list(frames(vid_data)), width=1440, height=1080, shape=1))
# # Add scatter plot with larger marker size
# scatter_trace = px.scatter(df_to_plot, x='pose_x', y='pose_y', size_max=15).data[0]
# scatter_trace2 = px.scatter(df_to_plot, x='rfid_x', y='rfid_y', size_max=15).data[0]
# scatter_trace.marker.size = 6  # Increase marker size
# scatter_trace.marker.color = 'red'  # Make markers more visible
# fig.add_trace(scatter_trace)
# fig.add_trace(scatter_trace2)
# # Update layout to make plot bigger and adjust margins
# fig.update_layout(
#     width=1200,
#     height=1000,
#     margin=dict(l=20, r=20, t=20, b=20),  # Reduce margins to use more space
#     showlegend=False
# )
# # Show the figure
# fig.show()

In [ ]:
# Calculate RFID reader ranges with social data 
# (this should be less accurate because a RFID reading could potentially be matched to the wrong subject if only 1 subject is detected by SLEAP)
quantiles = [0.95, 0.99, 1.0]
social_reader_ranges = (
    social_matched_df
    .dropna(subset=['rfid_pose_distance'])
    .groupby('rfid_reader_name')['rfid_pose_distance']
    .agg(['median', 'mean', ('p95', lambda x: x.quantile(0.95)), 
          ('p99', lambda x: x.quantile(0.99)), 'max'])
    .round(1)
)

print("RFID Reader Ranges (cm):")
print(social_reader_ranges)

In [ ]:
def make_rfid_visits(
    pos_df: pd.DataFrame,
    matched_df: pd.DataFrame,
    reader_ranges: pd.DataFrame,
    rfid_devices_loc,
    cm2px: float,
    threshold_col: str = 'p99',
    min_duration: float = 1.0,
    gap_seconds: float = 1.0
) -> pd.DataFrame:
    """
    Returns a DataFrame of RFID 'visits' (mouse stays within range > min_duration)
    and the delay to the first RFID read *during that visit*.
    """
    # 1) get static reader positions & thresholds
    reader_coords = {}
    for name, loc in rfid_devices_loc.items():
        if (name.startswith('_') or name.startswith('*') or
            callable(getattr(rfid_devices_loc, name, None)) or
            name in ['to_pandas','to_dict','toDict','empty','copy']):
            continue
        if hasattr(loc, 'X') and hasattr(loc, 'Y'):
            reader_coords[name] = (float(loc.X), float(loc.Y))
    thresholds_px = {
        reader: reader_ranges.loc[reader, threshold_col] * cm2px
        for reader in reader_coords
        if threshold_col in reader_ranges.columns
    }

    # 2) collect all frames where mouse is within px radius of any reader
    fragments = []
    for reader, (rx, ry) in reader_coords.items():
        thresh_px = thresholds_px.get(reader)
        if thresh_px is None:
            continue
        # squared test in px
        rad2 = thresh_px ** 2
        mask = ((pos_df.x - rx) ** 2 + (pos_df.y - ry) ** 2) <= rad2
        if not mask.any():
            continue
        tmp = pos_df.loc[mask, ['identity_name']].copy()
        tmp['reader_name'] = reader
        tmp['time']        = tmp.index
        fragments.append(tmp)

    vc = pd.concat(fragments, ignore_index=True)

    # 3) split into visits (gap > gap_seconds breaks)
    vc.sort_values(['identity_name','reader_name','time'], inplace=True)
    vc['dt']        = vc.groupby(['identity_name','reader_name'])['time'].diff()
    vc['new_visit'] = (vc['dt'] > pd.Timedelta(seconds=gap_seconds)).astype(int)
    vc['visit_id']  = vc.groupby(['identity_name','reader_name'])['new_visit'].cumsum()

    visits = (
        vc.groupby(['identity_name','reader_name','visit_id'])
          .agg(
            start_time=('time','min'),
            end_time  =('time','max'),
            n_frames  =('time','count')
          )
          .reset_index()
    )
    visits['duration_s'] = (
        visits['end_time'] - visits['start_time']
    ).dt.total_seconds()
    visits_df = visits[visits['duration_s'] >= min_duration].drop(columns='visit_id')

    # 4) prepare sorted RFID‐read times per (identity, reader)
    md = (
        matched_df
          .rename(columns={
            'rfid_identity':    'identity_name',
            'rfid_reader_name': 'reader_name'
          })[['identity_name','reader_name','rfid_time']]
          .dropna(subset=['identity_name','reader_name','rfid_time'])
    )
    rfid_times = {
        (iden, rd): np.sort(grp['rfid_time'].values.astype('datetime64[ns]'))
        for (iden, rd), grp in md.groupby(['identity_name','reader_name'])
    }

    # 5) for each visit, binary-search the first read in [start_time, end_time]
    visits_df = visits_df.reset_index(drop=True)
    delays = np.full(len(visits_df), np.nan)

    idx_groups = visits_df.groupby(['identity_name','reader_name']).indices
    for (iden, reader), idxs in idx_groups.items():
        starts = visits_df.loc[idxs, 'start_time'].values.astype('datetime64[ns]')
        ends   = visits_df.loc[idxs,   'end_time'  ].values.astype('datetime64[ns]')
        arr    = rfid_times.get((iden, reader))
        if arr is None or arr.size == 0:
            continue

        positions = np.searchsorted(arr, starts, side='left')
        in_bounds = positions < len(arr)
        if not in_bounds.any():
            continue

        valid_pos  = positions[in_bounds]
        valid_ends = ends[in_bounds]
        within     = arr[valid_pos] <= valid_ends

        if within.any():
            sel_idxs = np.array(idxs)[in_bounds][within]
            delays[sel_idxs] = (
                (arr[valid_pos[within]] - starts[in_bounds][within])
                / np.timedelta64(1, 's')
            ).astype(float)

    visits_df['first_detection_delay_s'] = delays
    return visits_df

visits = make_rfid_visits(
    pos_df = pre_post_social_position_df,
    matched_df = pre_post_social_matched_df,
    reader_ranges = pre_post_social_reader_ranges,
    rfid_devices_loc = rfid_devices_loc,
    cm2px = cm2px,
    threshold_col = 'mean',
    min_duration = 1.0,
    gap_seconds = 1.0
)
display(visits)
display(visits.groupby('reader_name').agg(n_visits=('start_time', 'count')).round(1))
median_latency = visits['first_detection_delay_s'].median()
accuracy = (len(visits) - visits['first_detection_delay_s'].isna().sum()) / len(visits)
print(f"Median latency: {median_latency:.2f} seconds")
print(f"Accuracy: {accuracy:.2%}")


In [ ]:
# Plot first_detection_delay_s distribution
fig = px.histogram(visits, x='first_detection_delay_s', nbins=100)
fig.update_traces(marker=dict(line=dict(width=1, color='black')))
fig.update_layout(
    title='First Detection Delay Distribution',
    xaxis_title='First Detection Delay (s)',
    yaxis_title='Count',
    width=800,
    height=600
)
fig.show()


In [ ]:
# Save results
file_path = save_dir / "rfid_results.json"
try:
    with open(file_path, 'r') as f:
        rfid_results = json.load(f)
except (FileNotFoundError, json.JSONDecodeError):
    rfid_results = []

# New result
new_result = {
    'name': experiment['name'],
    'mean_range': mean_range,
    'median_latency': median_latency,
    'accuracy': accuracy
}

# Update if name exists, otherwise append
updated = False
for i, result in enumerate(rfid_results):
    if result['name'] == new_result['name']:
        rfid_results[i] = new_result  # Overwrite existing entry
        updated = True
        break

if not updated:
    rfid_results.append(new_result)

# # Save back to file
# with open(file_path, 'w') as f:
#     json.dump(rfid_results, f, indent=2)

print(rfid_results)

# Compute means
mean_mean_range = np.mean([r['mean_range'] for r in rfid_results])
mean_median_latency = np.mean([r['median_latency'] for r in rfid_results])
mean_accuracy = np.mean([r['accuracy'] for r in rfid_results])

# Compute standard deviations
sd_mean_range = np.std([r['mean_range'] for r in rfid_results], ddof=1)
sd_median_latency = np.std([r['median_latency'] for r in rfid_results], ddof=1)
sd_accuracy = np.std([r['accuracy'] for r in rfid_results], ddof=1)

# Compute standard errors
n = len(rfid_results)
sem_mean_range = sd_mean_range / np.sqrt(n)
sem_median_latency = sd_median_latency / np.sqrt(n)
sem_accuracy = sd_accuracy / np.sqrt(n)

# Print summary statistics
print(f"Mean Range: {mean_mean_range:.1f} cm (SEM: {sem_mean_range:.1f})")
print(f"Mean Latency: {mean_median_latency:.2f} s (SEM: {sem_median_latency:.2f})")
print(f"Mean Accuracy: {mean_accuracy:.2%} (SEM: {sem_accuracy:.2%})")

# Create subplot layout
fig = make_subplots(rows=1, cols=3, shared_yaxes=False,
                    subplot_titles=["", "", ""], horizontal_spacing=0.1)

# Add Mean Range with error bar
fig.add_trace(go.Bar(
    x=["Mean Range (cm)"],
    y=[mean_mean_range],
    error_y=dict(type='data', array=[sem_mean_range]),
    name="Mean Range"
), row=1, col=1)

# Add Median Latency with error bar
fig.add_trace(go.Bar(
    x=["Mean Latency (s)"],
    y=[mean_median_latency],
    error_y=dict(type='data', array=[sem_median_latency]),
    name="Mean Latency"
), row=1, col=2)

# Add Accuracy with error bar
fig.add_trace(go.Bar(
    x=["Mean Accuracy (%)"],
    y=[mean_accuracy],
    error_y=dict(type='data', array=[sem_accuracy]),
    name="Mean Accuracy"
), row=1, col=3)

# Clean up layout
fig.update_layout(
    showlegend=False,
    width=900,
    height=400,
    title=None,
    margin=dict(t=20, b=20),
)

fig.update_xaxes(title_text="", showticklabels=True)
fig.update_yaxes(title_text="")

fig.show()

# svg_path = save_dir / "rfid_range_latency_accuracy.svg"
# pio.write_image(fig, str(svg_path), format="svg")

In [ ]:
# Plot RFID reader reads

# Group data by rfid_reader_name
reader_names = pre_post_social_matched_df['rfid_reader_name'].unique()

# Create a figure
time = pd.Timestamp(experiment["social_start"]) + pd.Timedelta(hours=3) + pd.Timedelta(minutes=22) + pd.Timedelta(seconds=1) # good for exp a4s4
root = Path(f"/ceph/aeon/aeon/data/raw/{acquisition_computer}/{social_name}")
vid_data = aeon_api.load(root, exp02.CameraTop.Video, start=time, end=time+pd.Timedelta(seconds=1/fps/2))
vid_data = vid_data[time:time] # idk why this is necessary by for some reason the video data is loading more than just the ts between time and time+1/fps/2
fig = px.imshow(gridframes(list(frames(vid_data)), width=1440, height=1080, shape=1))

# 1) get unique readers & their static positions
readers_df = (
    pre_post_social_matched_df
    .groupby('rfid_reader_name')
    .agg(rfid_x=('rfid_x','first'),
         rfid_y=('rfid_y','first'))
    .reset_index()
)

reader_names = readers_df['rfid_reader_name'].tolist()
n = len(reader_names)

# 2) pick a qualitative palette with enough distinct colors
base_palette = px.colors.qualitative.Plotly
# if you have more readers than the palette length, you can cycle:
reader_colors = [base_palette[i % len(base_palette)] for i in range(n)]
subject_color = [base_palette[(n) % len(base_palette)]]
colors = reader_colors + subject_color

# 3) loop over readers and add scatter traces
for i, reader in enumerate(reader_names):
    col = colors[i]
    # filter out rows for this reader
    sub = pre_post_social_matched_df[pre_post_social_matched_df['rfid_reader_name'] == reader]
    sub = sub.sample(n=min(5000, len(sub)), random_state=4)

    # a) subject detections
    fig.add_trace(
        go.Scatter(
            x=sub['pose_x'],
            y=sub['pose_y'],
            mode='markers',
            marker=dict(
                color=col,
                size=6,
                opacity=0.3
            ),
            name=f"{reader} detections"
        )
    )

    # b) reader static position
    rx = readers_df.loc[readers_df['rfid_reader_name']==reader, 'rfid_x'].item()
    ry = readers_df.loc[readers_df['rfid_reader_name']==reader, 'rfid_y'].item()
    col = colors[i]

    # darker but still saturated
    dark_col = px.colors.label_rgb(px.colors.hex_to_rgb(col))

    fig.add_trace(
        go.Scatter(
            x=[rx],
            y=[ry],
            mode='markers+text',
            marker=dict(
                symbol='x',   
                size=10,    
                color="black",
                opacity=1.0,
            ),
            text=[reader],
            textposition='top center',
            textfont=dict(
                size=14,
                family='Arial Black',
                color="black"
            ),
            showlegend=False
        )
    )

# 4) Add mice tracks
track_window = pd.Timedelta(seconds=0.5)
start_time = time - track_window
end_time = time

# Filter by timestamp index
track_df = social_position_df.loc[start_time:end_time]

# Get unique mice
subjects = track_df['identity_name'].unique()

for i, subject in enumerate(subjects):
    sub = track_df[track_df['identity_name'] == subject].sort_index()
    if len(sub) < 2:
        continue

    # Normalize time for fading effect
    timestamps = (sub.index - sub.index.min()).total_seconds()
    norm_time = timestamps / timestamps.max()

    fig.add_trace(
        go.Scatter(
            x=sub['x'],
            y=sub['y'],
            mode='lines',
            line=dict(
                color=colors[n],
                width=2,
            ),
            name=f"{subject} trail",
            hoverinfo='name+x+y',
            showlegend=True
        )
    )

# 5. Plot skeletons on top of video
# Load the pose data
r = aeon_reader.Pose(pattern="CameraTop_222*")
poses = aeon_api.load(
    str(root).replace("raw", "ingest"),
    r,
    start=time,
    end=time + pd.Timedelta(seconds=1/fps/2)
)

# Define skeleton connections
skeleton_edges = [
    ("nose", "head"),
    ("head", "right_ear"),
    ("head", "left_ear"),
    ("head", "spine1"),
    ("spine1", "spine2"),
    ("spine2", "spine3"),
    ("spine3", "spine4")
]
skeleton_parts = set([p for edge in skeleton_edges for p in edge])

# Plot skeletons for each mouse
for i, subject in enumerate(subjects):
    df_identity = poses[poses["identity"] == subject]
    if df_identity.empty:
        continue

    parts_dict = {
        part: grp.iloc[0]
        for part, grp in df_identity.groupby("part")
        if part in skeleton_parts
    }

    # Draw skeleton lines and markers
    for part_a, part_b in skeleton_edges:
        if part_a in parts_dict and part_b in parts_dict:
            a, b = parts_dict[part_a], parts_dict[part_b]
            if pd.notna(a["x"]) and pd.notna(b["x"]):
                # Line between parts
                fig.add_trace(
                    go.Scatter(
                        x=[a["x"], b["x"]],
                        y=[a["y"], b["y"]],
                        mode="lines",
                        line=dict(color=colors[n], width=3),
                        showlegend=False,
                        hoverinfo="none"
                    )
                )

    for part, row in parts_dict.items():
        if pd.notna(row["x"]) and pd.notna(row["y"]):
            # Part marker
            fig.add_trace(
                go.Scatter(
                    x=[row["x"]],
                    y=[row["y"]],
                    mode="markers",
                    marker=dict(
                        color=colors[n],
                        size=6,
                        line=dict(color="black", width=0.5),
                    ),
                    showlegend=False,
                    hoverinfo="text",
                    text=f"{subject} - {part}"
                )
            )


# now update layout as you already have
fig.update_layout(
    width=1200,
    height=1000,
    margin=dict(l=20, r=20, t=20, b=20),
)

fig.show()

# svg_path = save_dir / "arena_rfid_and_tracks.svg"
# pio.write_image(fig, str(svg_path), format="svg")

### 2. SLEAP accuracy

In [ ]:
# Extract date from rfid_time
social_matched_df['date'] = pd.to_datetime(social_matched_df['rfid_time']).dt.date

# Group by date and calculate metrics
results = []
for date, group in social_matched_df.groupby('date'):
    # Total rows for this day
    total = len(group)
    
    # Rows where SLEAP made a prediction
    predicted = group['pose_identity'].notna().sum()
    
    # Rows where prediction matches ground truth (only consider rows with predictions)
    correct = ((group['pose_identity'] == group['rfid_identity']) & 
               group['pose_identity'].notna()).sum()
    
    # Calculate metrics
    id_accuracy = correct / predicted if predicted > 0 else 0
    missed_rate = (total - predicted) / total
    
    results.append({
        'date': date,
        'id_accuracy': id_accuracy,
        'missed_rate': missed_rate
    })

# Convert to DataFrame for plotting
results_df = pd.DataFrame(results).sort_values('date')
if experiment["name"] == "social0.3-aeon3":
    results_df["id_accuracy"] = 1 - results_df["id_accuracy"]  # Invert accuracy for this experiment

# Create a single figure with both metrics
fig = go.Figure()

# Add ID accuracy trace
fig.add_trace(
    go.Scatter(
        x=results_df['date'], 
        y=results_df['id_accuracy'],
        mode='lines+markers',
        name='ID Accuracy',
        line=dict(color='blue')
    )
)

# Add missed prediction rate trace
# fig.add_trace(
#     go.Scatter(
#         x=results_df['date'], 
#         y=results_df['missed_rate'],
#         mode='lines+markers',
#         name='Missed Prediction Rate',
#         line=dict(color='red')
#     )
# )

# Update layout
fig.update_layout(
    height=500,
    width=900,
    title="SLEAP Performance Metrics by Day",
    xaxis_title="Date",
    yaxis_title= "ID Accuracy", #"Rate",
    yaxis=dict(range=[0, 1], tickformat='.0%'),
    showlegend=False #True
)

# Show figure
fig.show()

# Print summary statistics
display(results_df)
print(f"Average ID Accuracy: {results_df['id_accuracy'].mean():.2%}")
print(f"Average Missed Prediction Rate: {results_df['missed_rate'].mean():.2%}")

In [ ]:
visits = make_rfid_visits(
    pos_df = social_position_df,
    matched_df = social_matched_df,
    reader_ranges = pre_post_social_reader_ranges,
    rfid_devices_loc = rfid_devices_loc,
    cm2px = cm2px,
    threshold_col = 'max',
    min_duration = 1.0,
    gap_seconds = 1.0
)

# 1) Downcast the ID/reader columns to categoricals for lower memory footprint
visits[['identity_name','reader_name']] = visits[['identity_name','reader_name']].astype('category')
social_matched_df[['rfid_reader_name','rfid_identity','pose_identity']] = (
    social_matched_df[['rfid_reader_name','rfid_identity','pose_identity']]
    .astype('category')
)

# 2) Keep only visits that last ≤ 2 minutes (120 s)
short_visits = visits.loc[visits['duration_s'] <= 120, ['identity_name','reader_name','start_time','end_time']]

# 3) Prepare for a fast time‐based join
sd = social_matched_df.sort_values('rfid_time')
sv = short_visits.rename(columns={
    'identity_name':'rfid_identity',
    'reader_name':'rfid_reader_name'
}).sort_values('start_time')

# 4) Merge-asof to attach the nearest ’start_time’ ≤ rfid_time
merged = pd.merge_asof(
    sd,
    sv,
    left_on='rfid_time',
    right_on='start_time',
    by=['rfid_identity','rfid_reader_name'],
    direction='backward'
)

# 5) Only keep rows where the rfid_time actually falls before the visit’s end_time
filtered = merged[merged['rfid_time'] <= merged['end_time']]

# 6) Compute per‐day totals, predictions, and correct hits in one groupby
filtered['date'] = filtered['rfid_time'].dt.date
filtered = filtered.assign(
    predicted = filtered['pose_identity'].notna(),
    correct   = (filtered['pose_identity'] == filtered['rfid_identity'])
)

agg = (
    filtered
    .groupby('date')
    .agg(
        total     = ('rfid_time','size'),
        predicted = ('predicted','sum'),
        correct   = ('correct','sum'),
    )
    .reset_index()
)

agg['id_accuracy']  = agg['correct']   / agg['predicted'].replace(0, pd.NA)
agg['missed_rate'] = (agg['total'] - agg['predicted']) / agg['total']

results_df = agg[['date','id_accuracy','missed_rate']].fillna(0)

# 7) Invert accuracy for that one special experiment
if experiment["name"] == "social0.3-aeon3":
    results_df['id_accuracy'] = 1 - results_df['id_accuracy']

# 8) Plot exactly as before
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=results_df['date'], 
    y=results_df['id_accuracy'],
    mode='lines+markers',
    name='ID Accuracy',
    line=dict(color='blue')
))
fig.update_layout(
    height=500, width=900,
    title="SLEAP Performance Metrics by Day",
    xaxis_title="Date",
    yaxis_title="ID Accuracy",
    yaxis=dict(range=[0,1],tickformat='.0%'),
    showlegend=False
)
fig.show()

# 9) Summary stats
display(results_df)
print(f"Average ID Accuracy: {results_df['id_accuracy'].mean():.2%}")
print(f"Average Missed Prediction Rate: {results_df['missed_rate'].mean():.2%}")

# svg_path = save_dir / "id_accuracy_over_time.svg"
# pio.write_image(fig, str(svg_path), format="svg")

In [ ]:
# # Optional debugging
# # User-defined: which day to analyze (1 = first day, 2 = second day, etc.)
# n = 12  # Change this value as needed

# # Convert rfid_time to datetime if not already
# social_matched_df['rfid_time'] = pd.to_datetime(social_matched_df['rfid_time'])

# # Extract date and hour
# social_matched_df['date'] = social_matched_df['rfid_time'].dt.date
# social_matched_df['hour'] = social_matched_df['rfid_time'].dt.hour

# # Sort unique dates and select the nth one
# unique_dates = sorted(social_matched_df['date'].unique())
# if n > len(unique_dates) or n < 1:
#     raise ValueError(f"Requested day {n} is out of range. Dataset has {len(unique_dates)} unique day(s).")
# selected_date = unique_dates[n - 1]

# # Filter for selected day
# day_df = social_matched_df[social_matched_df['date'] == selected_date]
# display(day_df)

# # Group by hour and compute ID accuracy
# hourly_results = []
# for hour, group in day_df.groupby('hour'):
#     predicted = group['pose_identity'].notna().sum()
#     correct = ((group['pose_identity'] == group['rfid_identity']) &
#                group['pose_identity'].notna()).sum()
#     id_accuracy = correct / predicted if predicted > 0 else 0
#     print(predicted, correct, id_accuracy)
    
#     hourly_results.append({
#         'hour': hour,
#         'id_accuracy': id_accuracy
#     })

# # Convert to DataFrame
# hourly_df = pd.DataFrame(hourly_results).sort_values('hour')

# # Plot
# fig = go.Figure()
# fig.add_trace(
#     go.Scatter(
#         x=hourly_df['hour'],
#         y=hourly_df['id_accuracy'],
#         mode='lines+markers',
#         name='ID Accuracy',
#         line=dict(color='blue')
#     )
# )

# # Update layout
# fig.update_layout(
#     height=500,
#     width=900,
#     title=f"ID Accuracy Hour by Hour on Day {n} ({selected_date})",
#     xaxis_title="Hour of Day",
#     yaxis_title="ID Accuracy",
#     yaxis=dict(range=[0, 1], tickformat='.0%'),
#     xaxis=dict(dtick=1),
#     showlegend=False
# )

# fig.show()


In [ ]:
# Save results
file_path = save_dir / "sleap_results_cleaned_swaps.json"
try:
    with open(file_path, 'r') as f:
        sleap_results = json.load(f)
except (FileNotFoundError, json.JSONDecodeError):
    sleap_results = []

# New result
new_result = {
    'name': experiment['name'],
    'median_id_accuracy': results_df['id_accuracy'].median(),
}

# Update if name exists, otherwise append
updated = False
for i, result in enumerate(sleap_results):
    if result['name'] == new_result['name']:
        sleap_results[i] = new_result # Overwrite existing entry
        updated = True
        break

if not updated:
    sleap_results.append(new_result)

# Save back to file
with open(file_path, 'w') as f:
    json.dump(sleap_results, f, indent=2)

print(sleap_results)

mean_median_id_accuracy = np.mean([r['median_id_accuracy'] for r in sleap_results])
sd_id_accuracy = np.std([r['median_id_accuracy'] for r in sleap_results], ddof=1)
sem_id_accuracy = sd_id_accuracy / np.sqrt(len(sleap_results))

# print summary statistics
print(f"Mean Median ID Accuracy: {mean_median_id_accuracy:.2%} (SEM: {sem_id_accuracy:.2%})")

# plot the single bar with error bar
fig = go.Figure()
fig.add_trace(go.Bar(
    x=["Mean ID Accuracy"],
    y=[mean_median_id_accuracy],
    error_y=dict(type='data', array=[sem_id_accuracy]),
    name="Mean ID Accuracy"
))
# Clean up layout
fig.update_layout(
    showlegend=False,
    width=300,
    height=400,
    title=None,
    margin=dict(t=20, b=20),
)
fig.update_xaxes(title_text="", showticklabels=True)
fig.update_yaxes(title_text="")
fig.show()

svg_path = save_dir / "mean_id_accuracy.svg"
pio.write_image(fig, str(svg_path), format="svg")

# Dominance plots

## Load data

In [ ]:
social_retreat_dfs = []

for exp in [experiments[i] for i in [0, 1, 4, 5]]:
    data = load_experiment_data(
        experiment=exp,
        data_dir=data_dir,
        periods=['social'],
        data_types=['retreat'],
        # trim_days=1  # Optional: trim
    )
    df = data['social_retreat']
    df['experiment_name'] = exp['name']
    social_retreat_dfs.append(df)

social_retreat_df = pd.concat(social_retreat_dfs, ignore_index=True)

**Tube test results, for reference:**

**SOCIAL 0.2**

_Pre-social tube test results:_
- BAA-1104045: 2, BAA-1104047: 8
- BAA-1104048: 7, BAA-1104049: 3

_Post-social tube test results:_
- BAA-1104045: 1, BAA-1104047: 9
- BAA-1104048: 8, BAA-1104049: 2

**SOCIAL 0.4**

_Pre-social tube test results:_
- BAA-1104795: 10, BAA-1104797: 4
- BAA-1104792: 4, BAA-1104794: 12

_Post-social tube test results:_
- BAA-1104795: 12, BAA-1104797: 3
- BAA-1104792: 2, BAA-1104794: 13

In [ ]:
# NOT SMOOTHED
# 1. Ensure 'end_timestamp' is datetime and extract 'date'
social_retreat_df['end_timestamp'] = pd.to_datetime(social_retreat_df['end_timestamp'])
social_retreat_df['date'] = social_retreat_df['end_timestamp'].dt.date

# 2. Loop over each experiment_name’s subset and build one plot per experiment
for exp_name, df_group in social_retreat_df.groupby('experiment_name'):
    # Compute daily win‐counts per mouse
    wins = (
        df_group
        .groupby(['date', 'winner_identity'])
        .size()
        .unstack(fill_value=0)
    )
    # Turn counts into proportions
    proportions = wins.div(wins.sum(axis=1), axis=0).reset_index()

    # Melt to long form for plotting
    df_long = proportions.melt(
        id_vars='date',
        var_name='winner_identity',
        value_name='proportion'
    )

    # Create and show one figure for this experiment
    fig = px.line(
        df_long,
        x='date',
        y='proportion',
        color='winner_identity',
        markers=True,
        title=f"Experiment: {exp_name}<br>Daily Proportion of Victories per Mouse"
    )
    fig.update_layout(
        xaxis_title='Date',
        yaxis_title='Proportion of Victories',
        legend_title='Mouse ID'
    )
    fig.show()


In [ ]:
# SMOOTHED

# Parameters
BIN_HOURS  = 6   # how many hours per bin (e.g. 6, 8, 10, 12, etc.)
SMOOTH_DAYS = 3   # how many days to smooth over (e.g. 3, 5, 7, etc.)

# 1. Ensure 'end_timestamp' is datetime
social_retreat_df['end_timestamp'] = pd.to_datetime(social_retreat_df['end_timestamp'])

# 2. Loop over each experiment_name’s subset and build one plot per experiment
for exp_name, df_group in social_retreat_df.groupby('experiment_name'):
    df_group = df_group.copy()

    # Compute bin start for each timestamp
    earliest_midnight = df_group['end_timestamp'].dt.normalize().min()
    bin_size = pd.Timedelta(hours=BIN_HOURS)
    offsets = df_group['end_timestamp'] - earliest_midnight
    n_bins = (offsets // bin_size).astype(int)
    df_group['bin_start'] = earliest_midnight + n_bins * bin_size

    # Compute win-counts per bin per mouse
    wins = (
        df_group
        .groupby(['bin_start', 'winner_identity'])
        .size()
        .unstack(fill_value=0)
    )

    # Drop bins with zero total wins
    wins = wins[wins.sum(axis=1) > 0]

    # Turn counts into proportions
    proportions = wins.div(wins.sum(axis=1), axis=0).reset_index()

    # Melt to long form for plotting
    df_long = proportions.melt(
        id_vars='bin_start',
        var_name='winner_identity',
        value_name='proportion'
    )

    # Compute moving average window size (in bins)
    window_size = int((SMOOTH_DAYS * 24) / BIN_HOURS)
    window_size = max(window_size, 1)

    # Apply centered moving average per mouse
    df_long = df_long.sort_values(['winner_identity', 'bin_start'])
    df_long['smoothed_prop'] = (
        df_long
        .groupby('winner_identity')['proportion']
        .transform(lambda s: s.rolling(window=window_size, center=True, min_periods=1).mean())
    )

    # Create and show one figure for this experiment
    fig = px.line(
        df_long,
        x='bin_start',
        y='smoothed_prop',
        color='winner_identity',
        markers=True,
        title=(f"Experiment: {exp_name}<br>"
               f"{BIN_HOURS}H Bins, {SMOOTH_DAYS}‐Day Moving Average")
    )
    fig.update_layout(
        xaxis_title=f"Bin Start (every {BIN_HOURS} hours from {earliest_midnight.date()})",
        yaxis_title=f"{SMOOTH_DAYS}‐Day MA of Win Proportion",
        legend_title='Mouse ID'
    )
    fig.show()


# Patch preference plots

## Load data

In [ ]:
data = load_experiment_data(
    data_dir=data_dir,
    data_types=['patch', 'patchinfo'],
)

patch_df = data['None_patch']
patch_info_df = data['None_patchinfo']

In [ ]:
def patch_info_df_to_dict(df):
    result_dict = {}
    for _, row in df.iterrows():
        experiment = row['experiment_name']
        
        # Initialize experiment entry if it doesn't exist
        if experiment not in result_dict:
            result_dict[experiment] = []
        
        # Create entry dictionary
        entry = {
            'block_start': row['block_start'].to_pydatetime(),
            'patch_name': row['patch_name'],
            'patch_rate': row['patch_rate'],
            'patch_offset': row['patch_offset'],
        }
        
        # Add entry to experiment list
        result_dict[experiment].append(entry)
    
    return result_dict

def patch_df_to_dict(df):
    results_dict = {}
    for experiment in df['experiment_name'].unique():
        # Filter the dataframe to only include rows for this experiment
        experiment_df = df[df['experiment_name'] == experiment].copy()
        
        # Remove the columns you don't want
        experiment_df = experiment_df.drop(columns=['experiment_name'])
        
        # Assign the filtered DataFrame directly to the dictionary
        results_dict[experiment] = experiment_df
    
    return results_dict

def get_first_half_social(df):
    # Get social data
    social_data = df[df['period']=='social']
    
    # Group social data by experiment
    social_by_exp = social_data.groupby('experiment_name')
    
    # For each experiment, take the first half of social data
    half_social_data_list = []
    
    for exp_name, exp_data in social_by_exp:
        # Sort by block_start to ensure chronological order
        exp_data_sorted = exp_data.sort_values('block_start')
        
        # Calculate the half-point
        half_point = len(exp_data_sorted) // 2
        
        # Take the first half
        first_half = exp_data_sorted.iloc[:half_point]
        
        # Add to our list
        half_social_data_list.append(first_half)
    
    # Combine all first halves into one dataframe
    half_social_data = pd.concat(half_social_data_list) if half_social_data_list else pd.DataFrame(columns=social_data.columns)
    
    return half_social_data

block_subject_patch_data_social_combined = patch_df[patch_df['period']=='social'].copy()
block_subject_patch_data_social_combined.drop(columns=['period'], inplace=True)
block_subject_patch_data_social_dict = patch_df_to_dict(block_subject_patch_data_social_combined)
block_subject_patch_data_post_social_combined = patch_df[patch_df['period']=='postsocial'].copy()
block_subject_patch_data_post_social_combined.drop(columns=['period'], inplace=True)
block_subject_patch_data_post_social_dict = patch_df_to_dict(block_subject_patch_data_post_social_combined)
block_subject_patch_data_social_first_half_combined = get_first_half_social(patch_df).copy()
block_subject_patch_data_social_first_half_combined.drop(columns=['period'], inplace=True)
block_subject_patch_data_social_first_half_dict = patch_df_to_dict(block_subject_patch_data_social_first_half_combined)
patch_info_dict = patch_info_df_to_dict(patch_info_df)

### 1. Wheel distance spun per block, averaged by the number of mice

In [ ]:
block_subject_patch_data_social_combined['final_wheel_cumsum'] = block_subject_patch_data_social_combined['wheel_cumsum_distance_travelled'].apply(lambda x: x[-1] if isinstance(x, np.ndarray) and len(x) > 0 else 0)
wheel_total_dist_averaged_social = block_subject_patch_data_social_combined.groupby('block_start')['final_wheel_cumsum'].sum() / 2
wheel_total_dist_averaged_social = wheel_total_dist_averaged_social.reset_index()

block_subject_patch_data_post_social_combined['final_wheel_cumsum'] = block_subject_patch_data_post_social_combined['wheel_cumsum_distance_travelled'].apply(lambda x: x[-1] if isinstance(x, np.ndarray) and len(x) > 0 else 0)
wheel_total_dist_averaged_post_social = block_subject_patch_data_post_social_combined.groupby('block_start')['final_wheel_cumsum'].sum().reset_index()

wheel_total_dist_averaged_social['condition'] = 'social'
wheel_total_dist_averaged_post_social['condition'] = 'post_social'
wheel_total_dist_averaged = pd.concat([wheel_total_dist_averaged_social, wheel_total_dist_averaged_post_social])

fig = go.Figure()

fig = px.box(
    wheel_total_dist_averaged,
    x="condition",
    y="final_wheel_cumsum",
    points="all",
    title="Wheel Distance Spun Per Block Averaged By Number Of Subjects",
    labels={"final_wheel_cumsum": "Wheel Distance Spun Per Block (cm)"},
)
fig.show()

### 2. Number of patch switches by each mouse per block

In [ ]:
def compute_patch_probabilities(
    df: pd.DataFrame
) -> pd.DataFrame:
    """
    Compute patch probabilities based on block and subject data.

    Args:
        df (pd.DataFrame): Input DataFrame containing block, subject, pellet, and patch data.

    Returns:
        pd.DataFrame: A DataFrame with the probabilities for each patch.
    """
    results = []

    # Precompute unique block-subject groups
    grouped_data = df.groupby(['block_start', 'subject_name'])

    for (block_start, subject_name), block_data in grouped_data:
        # Process pellet timestamps once
        # Extract and ensure all pellet timestamps are float
        all_pellet_timestamps = []
        for sublist in block_data['pellet_timestamps']:
            for ts in sublist:
                try:
                    all_pellet_timestamps.append(float(ts))
                except (ValueError, TypeError):
                    # Skip values that can't be converted to float
                    pass
        
        pellet_timestamps = np.sort(np.unique(all_pellet_timestamps))
        if len(pellet_timestamps) < 2:
            continue

        # Create pellet intervals DataFrame
        intervals_df = pd.DataFrame({
            'interval_start': pellet_timestamps[:-1],
            'interval_end': pellet_timestamps[1:],
            'pellet_number': np.arange(1, len(pellet_timestamps))
        })

        # Prepare a dict to hold in_patch_timestamps for each patch
        patches_data = {}
        for patch in block_data['patch_name'].unique():
            patch_data = block_data[block_data['patch_name'] == patch]
            if patch_data.shape[0] != 1:
                raise ValueError("More than one row per block start, subject, patch combination.")
            
            # Get timestamps and ensure they're floats
            in_patch_ts_raw = patch_data.iloc[0]['in_patch_timestamps']
            in_patch_timestamps = []
            for ts in in_patch_ts_raw:
                try:
                    in_patch_timestamps.append(float(ts))
                except (ValueError, TypeError):
                    # Skip values that can't be converted to float
                    pass
            
            patches_data[patch] = np.sort(np.array(in_patch_timestamps))

        # Initialize a DataFrame to store counts per patch
        counts_df = intervals_df[['pellet_number']].copy()

        # For each patch, compute counts within each interval using numpy searchsorted
        for patch, in_patch_ts in patches_data.items():
            counts = np.zeros(len(intervals_df), dtype=int)
            if len(in_patch_ts) > 0:
                # Convert to float arrays explicitly
                in_patch_ts = in_patch_ts.astype(np.float64)
                interval_starts = intervals_df['interval_start'].values.astype(np.float64)
                interval_ends = intervals_df['interval_end'].values.astype(np.float64)
                
                idx_start = np.searchsorted(in_patch_ts, interval_starts, side='left')
                idx_end = np.searchsorted(in_patch_ts, interval_ends, side='right')
                counts = idx_end - idx_start
            counts_df[f'count_in_{patch}'] = counts

        # Compute total counts per interval
        counts_df['total_counts'] = counts_df.filter(like='count_in_').sum(axis=1)

        # Avoid division by zero
        counts_df['total_counts'] = counts_df['total_counts'].replace(0, np.nan)

        # Compute probabilities per interval
        for idx, row in counts_df.iterrows():
            pellet_number = row['pellet_number']
            row_data = {
                'block_start': block_start,
                'subject_name': subject_name,
                'pellet_number': pellet_number
            }
            ts_in_patches = {patch: row[f'count_in_{patch}'] for patch in patches_data.keys()}
            ts_in_patches_total = row['total_counts']
            if pd.isna(ts_in_patches_total):
                prob = {patch: 0 for patch in ts_in_patches.keys()}
            else:
                prob = {patch: ts_in_patches[patch] / ts_in_patches_total for patch in ts_in_patches.keys()}
            row_data.update({f'prob_in_{patch}': prob[patch] for patch in patches_data.keys()})
            results.append(row_data)

    # Create final DataFrame
    prob_per_patch = pd.DataFrame(results)
    return prob_per_patch

def extract_hard_patch_probabilities(
    prob_per_patch: pd.DataFrame, 
    patch_info: List[Dict[str, Any]], 
    patch_rate: float = 0.002
) -> pd.DataFrame:
    """
    Compute the probabilities for hard patches where the patch rate matches a specified value.

    Args:
        prob_per_patch (pd.DataFrame): DataFrame containing probabilities per patch.
        patch_info (List[Dict[str, Any]]): List of dictionaries with patch information.
        patch_rate (float): The patch rate to filter by. Default is 0.002.

    Returns:
        pd.DataFrame: A DataFrame with the probabilities for each hard patch.
    """
    # Filter the hard patches based on the patch_rate
    hard_patches = [patch_dict for patch_dict in patch_info if patch_dict['patch_rate'] == patch_rate]
    
    results = []
    for hard_patch in hard_patches:
        block_start = hard_patch['block_start']
        patch_name = hard_patch['patch_name']
        
        # Extract the hard patch data
        hard_patch_data = prob_per_patch.loc[
            prob_per_patch['block_start'] == block_start, 
            ['block_start', 'subject_name', 'pellet_number', f'prob_in_{patch_name}']
        ]
        
        # Rename the column for hard patch probability
        hard_patch_data = hard_patch_data.rename(columns={f'prob_in_{patch_name}': 'prob_in_hard_patch'})
        
        # Append the result to the list
        results.append(hard_patch_data)
        
    # Concatenate all results into a single DataFrame
    prob_hard_patch = pd.concat(results, ignore_index=True)
    
    return prob_hard_patch


In [ ]:
prob_per_patch_social_first_half_dict = {}
prob_per_patch_social_dict = {}
prob_per_patch_post_social_dict = {}
prob_hard_patch_social_first_half_dict = {}
prob_hard_patch_social_dict = {}
prob_hard_patch_post_social_dict = {}
prob_hard_patch_mean_social_first_half_dict = {}
prob_hard_patch_mean_social_dict = {}
prob_hard_patch_mean_post_social_dict = {}

for exp in experiments:
    exp_name = exp["name"]
    block_subject_patch_data_social_first_half = block_subject_patch_data_social_first_half_dict[exp_name]
    block_subject_patch_data_social = block_subject_patch_data_social_dict[exp_name]
    block_subject_patch_data_post_social = block_subject_patch_data_post_social_dict[exp_name]
    patch_info = patch_info_dict[exp_name]

    # Compute patch probabilities
    prob_per_patch_social_first_half_dict[exp_name] = compute_patch_probabilities(block_subject_patch_data_social_first_half)
    prob_per_patch_social_dict[exp_name] = compute_patch_probabilities(block_subject_patch_data_social)
    prob_per_patch_post_social_dict[exp_name] = compute_patch_probabilities(block_subject_patch_data_post_social)

    # Extract hard patch probabilities
    prob_hard_patch_social_first_half_dict[exp_name] = extract_hard_patch_probabilities(prob_per_patch_social_first_half_dict[exp_name], patch_info)
    prob_hard_patch_social_dict[exp_name] = extract_hard_patch_probabilities(prob_per_patch_social_dict[exp_name], patch_info)
    prob_hard_patch_post_social_dict[exp_name] = extract_hard_patch_probabilities(prob_per_patch_post_social_dict[exp_name], patch_info)

    # Calculate the mean hard patch probability per pellet number
    prob_hard_patch_mean_social_first_half_dict[exp_name] = prob_hard_patch_social_first_half_dict[exp_name].groupby('pellet_number').mean(numeric_only=True).reset_index()
    prob_hard_patch_mean_social_dict[exp_name] = prob_hard_patch_social_dict[exp_name].groupby('pellet_number').mean(numeric_only=True).reset_index()
    prob_hard_patch_mean_post_social_dict[exp_name] = prob_hard_patch_post_social_dict[exp_name].groupby('pellet_number').mean(numeric_only=True).reset_index()

# Combine the results
prob_hard_patch_social_first_half_combined = pd.concat(prob_hard_patch_social_first_half_dict.values())
prob_hard_patch_social_combined = pd.concat(prob_hard_patch_social_dict.values())
prob_hard_patch_post_social_combined = pd.concat(prob_hard_patch_post_social_dict.values())

prob_hard_patch_mean_social_first_half_combined = prob_hard_patch_social_first_half_combined.groupby('pellet_number').mean(numeric_only=True).reset_index()
prob_hard_patch_mean_social_combined = prob_hard_patch_social_combined.groupby('pellet_number').mean(numeric_only=True).reset_index()
prob_hard_patch_mean_post_social_combined = prob_hard_patch_post_social_combined.groupby('pellet_number').mean(numeric_only=True).reset_index()

In [ ]:
def analyze_patch_probabilities(
    data: pd.DataFrame, 
    label: str
) -> Tuple[sm.OLS, np.ndarray]:
    """
    Analyze patch probabilities using a linear regression model.

    Args:
        data (pd.DataFrame): DataFrame containing the data to analyze.
        label (str): Label for the analysis.

    Returns:
        tuple: A tuple containing:
            - model (sm.OLS): The fitted linear regression model.
            - y_pred (np.ndarray): The predicted values from the model.
    """
    # Prepare the data for statsmodels (add a constant for the intercept)
    X = np.array(data['pellet_number'][0:35])
    y = np.array(data['prob_in_hard_patch'][0:35])
    # Add a constant to the independent variable X to calculate the intercept
    X_with_constant = sm.add_constant(X)
    # Fit the model using statsmodels
    model = sm.OLS(y, X_with_constant).fit()
    y_pred = model.predict(X_with_constant)
    # Get the p-value for the slope (it's the second value in pvalues)
    p_value = model.pvalues[1]
    print(f"P-value for the {label} slope: {p_value}")
    # Print full statistical summary
    print(f"{label} model summary: ", model.summary())
    return model, y_pred

model_social_first_half, y_pred_social_first_half = analyze_patch_probabilities(prob_hard_patch_mean_social_first_half_combined, "social first half")
model_social, y_pred_social = analyze_patch_probabilities(prob_hard_patch_mean_social_combined, "social")
model_post_social, y_pred_post_social = analyze_patch_probabilities(prob_hard_patch_mean_post_social_combined, "post-social")

In [ ]:
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=prob_hard_patch_mean_social_first_half_combined['pellet_number'], #[0:35],
    y=prob_hard_patch_mean_social_first_half_combined['prob_in_hard_patch'], #[0:35],
    mode='lines',
    name='First Half of Social Data',
    marker=dict(color='blue')
))
    
fig.add_trace(go.Scatter(
    x=prob_hard_patch_mean_social_combined['pellet_number'], #[0:35],
    y=prob_hard_patch_mean_social_combined['prob_in_hard_patch'], #[0:35],
    mode='lines',
    name='Social Data',
    marker=dict(color='red')
))

fig.add_trace(go.Scatter(
    x=prob_hard_patch_mean_post_social_combined['pellet_number'], #[0:35],
    y=prob_hard_patch_mean_post_social_combined['prob_in_hard_patch'], #[0:35],
    mode='lines',
    name='Post Social Data',
    marker=dict(color='#00CC96')
))

fig.add_trace(go.Scatter(
    x=prob_hard_patch_mean_social_first_half_combined['pellet_number'][0:35],
    y=y_pred_social_first_half,
    mode='lines',
    name='Social First Half Linear Regression Line',
    line=dict(dash='dash'),
    marker=dict(color='blue')  # Optional: to make the regression line dashed
))

fig.add_trace(go.Scatter(
    x=prob_hard_patch_mean_social_combined['pellet_number'][0:35],
    y=y_pred_social,
    mode='lines',
    name='Social Linear Regression Line',
    line=dict(dash='dash'),
    marker=dict(color='red')  # Optional: to make the regression line dashed
))

fig.add_trace(go.Scatter(
    x=prob_hard_patch_mean_social_combined['pellet_number'][0:35],
    y=y_pred_post_social,
    mode='lines',
    name='Post Social Linear Regression Line',
    line=dict(dash='dash'),  # Optional: to make the regression line dashed
    marker=dict(color='#00CC96')
))

fig.update_layout(
    title='Probability of being in hard patch over time',
    xaxis_title='Pellet number in block',
    yaxis_title='Hard patch probability'
)

fig.show()

In [ ]:
# Save the figure as an SVG file
# fig.write_image("hard_patch_probability.svg")

In [ ]:
# Define the number of rows and columns for the subplot grid
num_experiments = len(experiments)
num_cols = 2  
num_rows = (num_experiments + num_cols - 1) // num_cols  

# Create a subplot grid
fig = make_subplots(rows=num_rows, cols=num_cols, subplot_titles=[exp["name"] for exp in experiments])

# Iterate over each experiment and add a plot to the grid
for i, exp in enumerate(experiments):
    exp_name = exp["name"]
    row = (i // num_cols) + 1
    col = (i % num_cols) + 1

    # Add the plot to the grid
    fig.add_trace(go.Scatter(
        x=prob_hard_patch_mean_social_first_half_dict[exp_name]['pellet_number'][0:35], 
        y=prob_hard_patch_mean_social_first_half_dict[exp_name]['prob_in_hard_patch'][0:35],
        mode='lines', 
        name='First Half of Social Data',
        marker=dict(color='blue'),
        showlegend=(i == 0)),
    row=row, col=col)

    fig.add_trace(go.Scatter(
        x=prob_hard_patch_mean_social_dict[exp_name]['pellet_number'][0:35], 
        y=prob_hard_patch_mean_social_dict[exp_name]['prob_in_hard_patch'][0:35],
        mode='lines', 
        name='Social Data',
        marker=dict(color='red'),
        showlegend=(i == 0)),
    row=row, col=col)

    fig.add_trace(go.Scatter(
        x=prob_hard_patch_mean_post_social_dict[exp_name]['pellet_number'][0:35], 
        y=prob_hard_patch_mean_post_social_dict[exp_name]['prob_in_hard_patch'][0:35],
        mode='lines', 
        name='Post Social Data',
        marker=dict(color='#00CC96'),
        showlegend=(i == 0)),
    row=row, col=col)


fig.update_layout(height=800, width=1000)
fig.show()

# Data overview plot

## Load data

In [ ]:
data = load_experiment_data(
    experiment=experiment,
    data_dir=data_dir,
    periods=['social'],
    data_types=['patch', 'position', 'foraging', 'weight'],
    # trim_days=1
)
social_patch_df = data['social_patch']
social_position_df = data['social_position']
social_foraging_df = data['social_foraging']
social_weight_df = data['social_weight']

### 0. Histogram of block durations

In [ ]:
blocks_dfs = []
for exp in [experiments[i] for i in [0, 1, 4, 5]]:
    blocks_df = (Block & {"experiment_name": exp["name"]}).fetch(format='frame')
    blocks_df = blocks_df.reset_index()
    blocks_df = blocks_df.rename(columns={'experiment_name': 'experiment', 'block_start': 'block_start_time'})
    blocks_dfs.append(blocks_df)
blocks = pd.concat(blocks_dfs, ignore_index=True)

In [ ]:
fig = px.histogram(
    blocks,
    x='block_duration_hr',
    nbins=30,
    histnorm='probability',
    title='Histogram of Block Durations',
    labels={'block_duration_hr': 'Duration (hr)', 'probability': 'Proportion'}
)

fig.update_layout(
    xaxis=dict(range=[1, 3]),
    bargap=0,
    plot_bgcolor='white',
    paper_bgcolor='white',
    yaxis_title='Proportion'
)

fig.show()

# svg_path = save_dir / "block_durations.svg"
# pio.write_image(fig, str(svg_path), format="svg")


### 1. Position heatmaps over time per subject

In [ ]:
# 1) Copy and sort dataframe
df = social_position_df.copy().sort_index()

# 2) Compute time-of-day and flag dark vs light
df['tod'] = (
    df.index.hour
    + df.index.minute / 60
    + df.index.second / 3600
    + df.index.microsecond / 1e6 / 3600
)
df['is_dark'] = (df['tod'] >= light_off) & (df['tod'] < light_on)

# 3) Detect light/dark flips and assign period IDs
first_dark = df.groupby('identity_name')['is_dark'].transform('first')
shifted = df.groupby('identity_name')['is_dark'].shift()
shifted = shifted.where(shifted.notna(), first_dark)
df['light_change'] = df['is_dark'] != shifted
df['light_id'] = (
    df.groupby('identity_name')['light_change']
      .cumsum().astype(int)
    + 1
)

# 4) Remove x, y detections outside of arena bounds
acquisition_computer = experiment["name"].split("-")[1].upper()
social_name = experiment["name"].split("-")[0]
metadata_root = f"/ceph/aeon/aeon/data/raw/{acquisition_computer}/{social_name}"
metadata_reader = social02.Metadata
metadata = aeon_api.load(
    metadata_root,
    metadata_reader
)['metadata'].iloc[0]
outer_radius = float(metadata.ActiveRegion.ArenaOuterRadius)+10
center_x = float(metadata.ActiveRegion.ArenaCenter.X)
center_y = float(metadata.ActiveRegion.ArenaCenter.Y)
nest_x1 = float(metadata.ActiveRegion.NestRegion.ArrayOfPoint[1].X)
nest_x2 = float(metadata.ActiveRegion.NestRegion.ArrayOfPoint[2].X)
nest_y1 = float(metadata.ActiveRegion.NestRegion.ArrayOfPoint[1].Y)
nest_y2 = float(metadata.ActiveRegion.NestRegion.ArrayOfPoint[2].Y)
dist2 = (df['x'] - center_x)**2 + (df['y'] - center_y)**2
inside_arena = dist2 <= outer_radius**2
nest_pts = metadata.ActiveRegion.NestRegion.ArrayOfPoint
x_pts = [float(pt.X) for pt in nest_pts]
y_pts = [float(pt.Y) for pt in nest_pts]
nx_min, nx_max = min(x_pts)-10, max(x_pts)+10
ny_min, ny_max = min(y_pts)-10, max(y_pts)+10
inside_nest = (
    df['x'].between(nx_min, nx_max) &
    df['y'].between(ny_min, ny_max)
)
df = df[ inside_arena | inside_nest ]

# 5) Clean ID swaps in the data
df_corrected = clean_swaps(df)
df_corrected.dropna(subset=['x', 'y'], inplace=True)

In [ ]:
# Save the original index name and reset to unique integer indices
original_index_name = df_corrected.index.name or 'time'
df_corrected = df_corrected.reset_index()

# Initialize speed column
df_corrected['speed'] = 0.0

# Calculate the speed of each subject based on position data
for subject in df_corrected['identity_name'].unique():
    subject_df = df_corrected[df_corrected['identity_name'] == subject].copy()
    subject_df.sort_values(original_index_name, inplace=True)  # Sort by time
    if subject_df.empty:
        continue

    # Calculate the difference in position and time
    dxy = subject_df[["x", "y"]].diff().values[1:]  # Skip the first row (NaN)
    dt_ms = subject_df[original_index_name].diff().dt.total_seconds().values[1:] * 1000  # Convert to milliseconds

    # Calculate speed in cm/s
    speed = np.linalg.norm(dxy, axis=1) / dt_ms * 1000 / cm2px  # Convert to cm/s
    subject_df['speed'] = np.concatenate(([0], speed))  # Add zero for the first row

    # Apply a running average filter
    k = np.ones(10) / 10  # Running avg filter kernel (10 frames)
    subject_df['speed'] = np.convolve(subject_df['speed'], k, mode='same')

    # Update the original dataframe (now with unique indices!)
    mask = df_corrected['identity_name'] == subject
    df_corrected.loc[mask, 'speed'] = subject_df['speed'].values

# Set the index back to time
df_corrected = df_corrected.set_index(original_index_name)

# Remove rows where speed is above 200cm/s
df_corrected_filtered = df_corrected[df_corrected['speed'] <= 250]

In [ ]:
# # Debugging
# data_to_plot = df_corrected.copy()
# idx = 300000 # Change this to the index of the frame you want to plot
# fps = 50
# time = data_to_plot.index[idx]
# df_to_plot = data_to_plot.loc[time:time]
# acquisition_computer = experiment["name"].split("-")[1].upper()
# social_name = experiment["name"].split("-")[0]
# raw_root = Path(f"/ceph/aeon/aeon/data/raw/{acquisition_computer}/{social_name}")
# vid_data = aeon_api.load(raw_root, exp02.CameraTop.Video, start=time, end=time+pd.Timedelta(seconds=2/fps)).iloc[0:1]
# fig = px.imshow(gridframes(list(frames(vid_data)), width=1440, height=1080, shape=1))
# # Add scatter plot with larger marker size
# scatter_fig = px.scatter(
#     df_to_plot,
#     x='x',
#     y='y',
#     color='identity_name',
#     size_max=15
# )
# for trace in scatter_fig.data:
#     trace.marker.size = 6
#     trace.marker.line = dict(color='red')  # optional for visibility
#     fig.add_trace(trace)
# # Update layout to make plot bigger and adjust margins
# fig.update_layout(
#     width=1200,
#     height=1000,
#     margin=dict(l=20, r=20, t=20, b=20),  # Reduce margins to use more space
#     showlegend=False
# )
# # Show the figure
# fig.show()

In [ ]:
# For visualising in the notebook and saving the plots as a single fig with each subplot as an image, run the following cell
# df = df_corrected.reset_index()

# dark_color = "#555555"
# light_color = "#CCCCCC"
# subjects = sorted(df['identity_name'].unique())
# n_subj = len(subjects)
# n_per = int(df['light_id'].max())

# # 2) set up subplots
# fig, axes = plt.subplots(
#     nrows=n_subj,
#     ncols=n_per,
#     figsize=(2*n_per, 2*n_subj),
#     sharex=True,
#     sharey=True,
#     squeeze=False,
# )

# # 3) loop over axes and draw
# for i, subj in enumerate(subjects):
#     for j in range(1, n_per+1):
#         ax = axes[i, j-1]
#         sub = df[(df.identity_name==subj) & (df.light_id==j)]
#         if sub.empty:
#             ax.set_axis_off()
#             continue

#         col = dark_color if sub.is_dark.iloc[0] else light_color

#         ax.plot(
#             sub.x,
#             sub.y,
#             linestyle='none',
#             marker='o',
#             markersize=1,
#             color=col,
#             lw=0.8,
#             rasterized=True,
#             antialiased=False
#         )
#         ax.set_aspect('equal', 'box')
#         ax.axis('off')

# # 4) scale bar in bottom-right
# last_ax = axes[-1, -1]
# length_px = 0.2 * 100 * cm2px
# xmin, xmax = last_ax.get_xlim()
# ymin, ymax = last_ax.get_ylim()
# x1 = xmax - 0.02*(xmax-xmin)
# x0 = x1 - length_px
# y0 = ymin + 0.02*(ymax-ymin)
# last_ax.plot([x0, x1], [y0, y0], 'k-', lw=2)
# last_ax.text(
#     (x0+x1)/2, y0 - 0.06*(ymax-ymin),
#     '0.2 m',
#     va='top', ha='center', fontsize=8, color='k'
# )

# plt.tight_layout()

# svg_path = save_dir / "position_maps.svg"
# plt.savefig(svg_path, format="svg", dpi=300)

# plt.show()

In [ ]:
# For saving the plots in detail, run the following cell
# 1) Define colors
dark_color = "#555555"
light_color = "#CCCCCC"

# 2) Get unique subjects and number of periods
subjects = sorted(df_corrected_filtered['identity_name'].unique())
n_subj = len(subjects)
n_per = int(df_corrected_filtered['light_id'].max())

# 3) Group DataFrame by (subject, period)
grouped = df_corrected_filtered.groupby(['identity_name', 'light_id'])

# 4) Threshold for stationary
stationary_thresh = -1  # Set to -1 to keep all points

# 5) Loop over each period and make one-column subplot (one row per subject)
for period in range(1, n_per + 1):
    # 5.1) Figure out if this period is dark or light (pick any available row)
    sample_key = next(((subj, period) for subj in subjects if (subj, period) in grouped.indices), None)
    if sample_key:
        is_dark_period = bool(grouped.get_group(sample_key)['is_dark'].iloc[0])
    else:
        is_dark_period = False

    # 5.2) Create a subplot grid: rows = number of subjects, cols = 1
    fig = make_subplots(
        rows=n_subj,
        cols=1,
        shared_xaxes=True,
        shared_yaxes=True,
        horizontal_spacing=0.01,
        vertical_spacing=0.01
    )

    for i, subj in enumerate(subjects):
        row = i + 1
        key = (subj, period)

        # 5.3) If there's data for this (subject, period), plot it
        if key in grouped.indices:
            sub = grouped.get_group(key).copy()

            # 5.4) Keep all moving points + first of each stationary block
            is_moving = sub['speed'] > stationary_thresh
            change_flag = is_moving.astype(int).diff().fillna(1).astype(bool)
            keep = is_moving | ((~is_moving) & change_flag)
            sub_ds = sub.loc[keep]

            color = dark_color if sub['is_dark'].iloc[0] else light_color

            fig.add_trace(
                go.Scatter(
                    x=sub_ds['x'],
                    y=sub_ds['y'],
                    mode='lines', #+markers
                    line=dict(width=1, color=color),
                    # marker=dict(size=1, color=color),
                    showlegend=False
                ),
                row=row,
                col=1
            )

        # 5.5) Hide grid lines and tick labels for every subplot (even if it's empty)
        fig.update_xaxes(showgrid=False, showticklabels=False, row=row, col=1)
        fig.update_yaxes(showgrid=False, showticklabels=False, scaleanchor="x", row=row, col=1)

    # 5.6) Draw scale bar in bottom subplot if any data exists for this period
    last_key = next(((subj, period) for subj in subjects if (subj, period) in grouped.indices), None)
    if last_key:
        sub_last = grouped.get_group(last_key)
        xmin, xmax = sub_last['x'].min(), sub_last['x'].max()
        ymin, ymax = sub_last['y'].min(), sub_last['y'].max()

        length_px = 0.2 * 100 * cm2px  # 0.2 m → 20 cm in pixels
        x1 = xmax - 0.02 * (xmax - xmin)
        x0 = x1 - length_px
        y0 = ymin + 0.02 * (ymax - ymin)

        subplot_index = n_subj  # bottom row
        if subplot_index == 1:
            xref = "x"
            yref = "y"
        else:
            xref = f"x{subplot_index}"
            yref = f"y{subplot_index}"

        fig.add_shape(
            type="line",
            x0=x0, x1=x1, y0=y0, y1=y0,
            line=dict(color="black", width=2),
            xref=xref, yref=yref
        )
        fig.add_annotation(
            x=(x0 + x1) / 2,
            y=y0 - 0.06 * (ymax - ymin),
            text="0.2 m",
            showarrow=False,
            xref=xref, yref=yref,
            font=dict(size=8, color="black"),
            xanchor="center",
            yanchor="top"
        )

    # 5.7) Set white background, figure size, and margins
    fig.update_layout(
        paper_bgcolor="white",
        plot_bgcolor="white",
        width=200,
        height=200 * n_subj,
        margin=dict(l=0, r=0, t=0, b=0)
    )

    # 5.8) Save as SVG named light# or dark#
    fname = f"{'dark' if is_dark_period else 'light'}{period}_just_lines_less_downsample_a3s0.2.eps"
    pio.write_image(fig, str(save_dir / fname), format="eps")
    print(f"Saved {save_dir / fname}")


### 2. Locomotion speed over time per subject

In [ ]:
df = df_corrected_filtered.copy()
df.index.name = "time"

dark_color = "#555555"
light_color = "#CCCCCC"
subjects = sorted(df['identity_name'].unique())

agg = (
    df[["identity_name", "speed"]]
      .groupby("identity_name")
      .resample("1s")
      .mean()
      .dropna()
      .reset_index()
)

fig = go.Figure()
for subj in subjects:
    sub = agg[agg["identity_name"] == subj]
    fig.add_trace(
        go.Scatter(
            x=sub["time"],
            y=sub["speed"],
            mode="lines",
            name=subj,
            line=dict(width=1),
            opacity=1
        )
    )

fig.update_layout(
    title="Speed over Time",
    xaxis_title="Time",
    yaxis_title="Speed",
    legend_title="Subject",
    plot_bgcolor="white"
)

# pio.write_image(
#     fig,
#     str(save_dir / "speed_over_time.svg"),
#     format="svg"
# )

fig.show()

### 3. Foraging bouts raster plot

In [ ]:
dark_color = "#555555"
subjects = sorted(social_foraging_df['subject'].unique())

fig = px.timeline(
    social_foraging_df,
    x_start="start",
    x_end="end",
    y="subject",
    hover_data=["n_pellets", "cum_wheel_dist"],
    category_orders={"subject": subjects}
)

# this fixes the bar‐opacity (px.timeline sets trace.opacity by default)
fig.update_traces(
    opacity=1,
    marker_color=dark_color,
    marker_line_color=dark_color,
    marker_line_width=1.5
)

fig.update_layout(
    template='simple_white',
    plot_bgcolor='white',
    margin=dict(l=150, r=20, t=20, b=20),
    height=max(100, len(subjects)*25 + 50),
    xaxis=dict(
        showgrid=False, zeroline=False,
        showline=False, ticks='', showticklabels=False
    ),
    yaxis=dict(
        showgrid=False, zeroline=False,
        showline=False, ticks='', showticklabels=True,
        title=''
    )
)

pio.write_image(
    fig,
    str(save_dir / "foraging_bouts_raster.svg"),
    format="svg"
)

fig.show()

### 4. Wheel distance spun 

#### Over time per subject patch

In [ ]:
dt_seconds = 0.02

fig = go.Figure()

# 2) build each trace (continuous + Δ>0.5 downsample)
max_y = 0
for (subject, patch), grp in social_patch_df.groupby(['subject_name', 'patch_name']):
    grp = grp.sort_values('block_start')
    total_n = sum(len(a) for a in grp.wheel_cumsum_distance_travelled)

    times = np.empty(total_n, dtype='datetime64[ns]')
    dists = np.empty(total_n, dtype=float)
    idx, offset = 0, 0.0

    for bs_val, arr in zip(grp.block_start, grp.wheel_cumsum_distance_travelled):
        arr = np.asarray(arr)
        n   = arr.size
        offs  = (np.arange(n) * dt_seconds * 1e9).astype('timedelta64[ns]')
        times[idx:idx+n] = np.datetime64(bs_val) + offs
        dists[idx:idx+n] = arr + offset
        offset += arr[-1]
        idx    += n

    max_y = max(max_y, dists.max())

    # downsample
    diffs = np.abs(np.diff(dists, prepend=dists[0]))
    mask  = diffs > 0.5
    mask[0] = True

    fig.add_trace(
        go.Scatter(
            x=times[mask],
            y=dists[mask]/100, # convert to meters
            mode='lines',
            name=f"{subject} — {patch}",
            line=dict(width=1.5)
        )
    )

# 3) hide all ticks and tick lines; keep only the y-axis title
fig.update_xaxes(
    showgrid=False,
    zeroline=False,
    showline=False,
    showticklabels=False,
    ticks=""
)
fig.update_yaxes(
    title_text="Distance spun on wheel (m)",
    showgrid=False,
    zeroline=False,
    showline=False,
    showticklabels=False,
    ticks=""
)

# 4) vertical scale bar: 250 m high at the right edge
fig.add_shape(
    type="line",
    xref="paper", x0=1.02, x1=1.02,    
    yref="y",     y0=0,    y1=250,
    line=dict(color="black", width=2)
)
fig.add_annotation(
    xref="paper", x=1.04,
    y=125,
    text="250",
    showarrow=False,
    xanchor="left",
    yanchor="middle",
    font=dict(size=10, color="black")
)

# 5) layout tweaks and show
fig.update_layout(
    template="simple_white",
    margin=dict(l=20, r=80, t=20, b=20),
    showlegend=True
)

pio.write_image(
    fig,
    str(save_dir / "wheel_dist_over_time.svg"),
    format="svg"
)

fig.show(config={
    'staticPlot': True,
})

#### Dummy patch vs normal patches

In [ ]:
patch_dfs = []

for exp in [experiments[i] for i in [4, 5]]:
    data = load_experiment_data(
        experiment=exp,
        data_dir=data_dir,
        data_types=['patch']
    )
    df = data['None_patch']
    patch_dfs.append(df)

patch_df_s4 = pd.concat(patch_dfs).sort_index()

In [ ]:
dt_seconds = 0.02
summary = []

# Step 1: Compute total distance spun per (subject, patch)
for (subject, patch), grp in patch_df_s4.groupby(['subject_name', 'patch_name']):
    total_distance = sum(np.asarray(w)[-1] for w in grp['wheel_cumsum_distance_travelled'] if len(w) > 0)
    summary.append({'subject': subject, 'patch': patch, 'distance': total_distance / 100})  # convert to meters

summary_df = pd.DataFrame(summary)

# Step 2: Pivot to subject x patch format
pivot_df = summary_df.pivot(index='subject', columns='patch', values='distance').fillna(0)

# Step 3: Per-subject values for PatchDummy1 and mean of other patches
patch_dummy1_vals = pivot_df.get('PatchDummy1', pd.Series(0, index=pivot_df.index))
other_patch_vals = pivot_df.drop(columns='PatchDummy1', errors='ignore').mean(axis=1)

# Step 4: Compute means and SEMs
means = [patch_dummy1_vals.mean(), other_patch_vals.mean()]
sems  = [patch_dummy1_vals.sem(),  other_patch_vals.sem()]
labels = ['PatchDummy1', 'Avg Other Patches']

# Step 5: Plot
fig = go.Figure(data=[
    go.Bar(
        x=labels,
        y=means,
        error_y=dict(type='data', array=sems, visible=True),
        marker_color=['#636EFA', '#EF553B']
    )
])

fig.update_layout(
    title='Mean Wheel Distance Spun ± SEM',
    yaxis_title='Distance (m, log)',
    xaxis_title='Patch',
    yaxis_type='log',
    template='simple_white',
    height=400
)

pio.write_image(
    fig,
    str(save_dir / "dummy_vs_normal_patches.svg"),
    format="svg"
)

fig.show()

In [ ]:
display(patch_dummy1_vals, other_patch_vals)

t_stat, p_val = ttest_rel(patch_dummy1_vals, other_patch_vals)
print(f"Paired t‐statistic = {t_stat:.3f},  p‐value = {p_val:.4f}")

### 5. Pellets raster plot

In [ ]:
df = (
    social_patch_df
    [['subject_name','patch_name','pellet_timestamps']]
    .explode('pellet_timestamps')
    .dropna(subset=['pellet_timestamps'])
)
df['pellet_timestamps'] = pd.to_datetime(df['pellet_timestamps'])
df['patch_idx'] = df['patch_name'].str.extract(r'(\d+)$').astype(int)

dark_color = "#555555"
subjects = sorted(df['subject_name'].unique())
n_subj = len(subjects)

fig = make_subplots(
    rows=n_subj, cols=1,
    shared_xaxes=True,
    subplot_titles=subjects
)

for i, subj in enumerate(subjects, start=1):
    sub = df[df['subject_name']==subj]
    fig.add_trace(
        go.Scatter(
            x=sub['pellet_timestamps'],
            y=sub['patch_idx'],
            mode='markers',
            marker=dict(symbol='line-ns',
                        color=dark_color,
                        size=8,
                        line_width=1.2),
            showlegend=False
        ),
        row=i, col=1
    )
    # y-axis = patch numbers
    fig.update_yaxes(
        row=i, col=1,
        title='Patches',
        tickmode='array',
        showgrid=False, zeroline=False, showline=False, ticks=''
    )
    # remove x-labels on every row
    fig.update_xaxes(
        row=i, col=1,
        showticklabels=False,
        showgrid=False, zeroline=False, showline=False, ticks=''
    )

# tighten the margins and overall height
fig.update_layout(
    template='simple_white',
    plot_bgcolor='white',
    margin=dict(l=80, r=20, t=80, b=20),
    height=120 * n_subj   # reduce per‐row height
)

# center your subject titles
for ann in fig.layout.annotations:
    ann.x = 0.5
    ann.xanchor = 'center'
    ann.font = dict(size=16)

pio.write_image(
    fig,
    str(save_dir / "pellets_raster.svg"),
    format="svg"
)

fig.show()

### 6. Weight over time

#### Per subject for a single experiment

In [ ]:
# 1. Keep only rows where you actually have timestamps
df = social_weight_df[social_weight_df['timestamps'].apply(lambda lst: len(lst) > 0)].copy()

# 2. Explode your parallel list-columns into long form
df = df.explode(['timestamps', 'weight', 'subject_id'])

# 3. Type conversions
df['timestamps'] = pd.to_datetime(df['timestamps'])
df['weight']     = df['weight'].astype(float)

# 4. Plot with Plotly Express
fig = px.line(
    df,
    x='timestamps',
    y='weight',
    color='subject_id',
    markers=True,
    title="Subject Weights Over Time",
    labels={
        'timestamps': 'Time',
        'weight': 'Weight',
        'subject_id': 'Subject ID'
    }
)
fig.update_layout(legend_title_text='Subject ID')
fig.show()

#### Averaged over time

In [ ]:
social_weight_dfs = []

for exp in [experiments[i] for i in [0, 1, 4, 5]]:
    data = load_experiment_data(
        experiment=exp,
        data_dir=data_dir,
        periods=['social'],
        data_types=['weight']
    )
    df = data['social_weight']
    social_weight_dfs.append(df)

social_weight_df_all_exps = pd.concat(social_weight_dfs).sort_index()

In [ ]:
# 1) Flatten the “array-of-arrays” df into a long DataFrame
records = []
for _, row in social_weight_df_all_exps.iterrows():
    ts = pd.to_datetime(row['timestamps'])
    w = np.asarray(row['weight'], dtype=float)
    sids = row['subject_id']
    for t, weight, sid in zip(ts, w, sids):
        records.append({
            'timestamp': t,
            'subject_id': sid,
            'weight': weight
        })

df = pd.DataFrame.from_records(records)
df = df.sort_values(['subject_id', 'timestamp'])
# drop rows where subject_id is shorter than 11 characters, removes some erroneous entries
df = df[df['subject_id'].str.len() >= 11].reset_index(drop=True)

# 2) Parameters & 24 h cycle grid
sampling_freq = '10min'  # resample interval
# choose any date at 08:00 to define “day zero”
anchor = pd.Timestamp(f'2020-01-01 {light_off:02d}:00:00')

# build the 24 h cycle index
n_steps = int(pd.Timedelta('1D') / pd.Timedelta(sampling_freq))
cycle_index = pd.timedelta_range(start=0, periods=n_steps, freq=sampling_freq)

# will hold each mouse’s mean‐day
cycle_df = pd.DataFrame(index=cycle_index)

# 3) Loop over each mouse, resample, fold into 24 h, average days
for sid, grp in df.groupby('subject_id'):
    # a) series of weight vs time
    ser = grp.set_index('timestamp')['weight'].sort_index()
    # b) collapse any exact-duplicate timestamps
    ser = ser.groupby(level=0).mean()

    # c) resample into bins anchored at 08:00, then interpolate
    ser_rs = (
        ser
        .resample(sampling_freq, origin=anchor)
        .mean()
        .interpolate()
    )

    # d) convert each timestamp into its offset (mod 24 h) from the anchor
    offsets = (ser_rs.index - anchor) % pd.Timedelta('1D')
    ser_rs.index = offsets

    # e) average across all days for each offset
    daily = ser_rs.groupby(ser_rs.index).mean()

    # f) align to our uniform cycle grid
    cycle_df[sid] = daily.reindex(cycle_index)

# 4) Baseline‐subtract each mouse’s minimum, then grand‐mean
cycle_df_baselined = cycle_df.subtract(cycle_df.min(skipna=True), axis=1)
grand_mean = cycle_df_baselined.mean(axis=1)
sem = cycle_df_baselined.sem(axis=1)

# 5) smooth both mean and SEM with a centered rolling window
window = 20
grand_mean_smooth = grand_mean.rolling(window=window, center=True, min_periods=1).mean()
sem_smooth = sem.rolling(window=window, center=True, min_periods=1).mean()

# 5) Plot each subject's mean-day curve in its own subplot
import math

n_subj = cycle_df_baselined.shape[1]
n_cols = 4  # adjust as needed
n_rows = math.ceil(n_subj / n_cols)

fig, axes = plt.subplots(n_rows, n_cols, figsize=(3.5 * n_cols, 2.5 * n_rows), sharex=True, sharey=True)
axes = axes.flatten()
x_hours = cycle_df_baselined.index.total_seconds() / 3600

for i, sid in enumerate(cycle_df_baselined.columns):
    ax = axes[i]
    y = cycle_df_baselined[sid]
    y_smooth = y.rolling(window=window, center=True, min_periods=1).mean()
    ax.plot(x_hours, y_smooth, lw=1.5)
    ax.set_title(sid, fontsize=9)
    ax.set_xlim(0, 24)
    ax.grid(True, linestyle=':', linewidth=0.5)

# Remove any unused axes
for ax in axes[n_subj:]:
    ax.set_visible(False)

fig.suptitle('24 h weight cycle (baseline-subtracted) per subject', y=1.02)
fig.text(0.5, 0.04, f'Time since {light_off:02d}:00:00 (hours)', ha='center')
fig.text(0.04, 0.5, 'Weight (baseline-subtracted)', va='center', rotation='vertical')
plt.subplots_adjust(hspace=0.4, wspace=0.3, bottom=0.1, left=0.07, right=0.97, top=0.90)
plt.show()

#### Averaged over time and subjects

Baselined by subtracting the minimum of each subject's 24h mean-day curve

In [ ]:
# 1) Flatten the “array-of-arrays” df into a long DataFrame
records = []
for _, row in social_weight_df_all_exps.iterrows():
    ts = pd.to_datetime(row['timestamps'])
    w = np.asarray(row['weight'], dtype=float)
    sids = row['subject_id']
    for t, weight, sid in zip(ts, w, sids):
        records.append({
            'timestamp': t,
            'subject_id': sid,
            'weight': weight
        })

df = pd.DataFrame.from_records(records)
df = df.sort_values(['subject_id', 'timestamp'])
# drop rows where subject_id is shorter than 11 characters, removes some erroneous entries
df = df[df['subject_id'].str.len() >= 11].reset_index(drop=True)

# 2) Parameters & 24 h cycle grid
sampling_freq = '10min'  # resample interval
# choose any date at 08:00 to define “day zero”
anchor = pd.Timestamp(f'2020-01-01 {light_off:02d}:00:00')

# build the 24 h cycle index
n_steps = int(pd.Timedelta('1D') / pd.Timedelta(sampling_freq))
cycle_index = pd.timedelta_range(start=0, periods=n_steps, freq=sampling_freq)

# will hold each mouse’s mean‐day
cycle_df = pd.DataFrame(index=cycle_index)

# 3) Loop over each mouse, resample, fold into 24 h, average days
for sid, grp in df.groupby('subject_id'):
    # a) series of weight vs time
    ser = grp.set_index('timestamp')['weight'].sort_index()
    # b) collapse any exact-duplicate timestamps
    ser = ser.groupby(level=0).mean()

    # c) resample into bins anchored at 08:00, then interpolate
    ser_rs = (
        ser
        .resample(sampling_freq, origin=anchor)
        .mean()
        .interpolate()
    )

    # d) convert each timestamp into its offset (mod 24 h) from the anchor
    offsets = (ser_rs.index - anchor) % pd.Timedelta('1D')
    ser_rs.index = offsets

    # e) average across all days for each offset
    daily = ser_rs.groupby(ser_rs.index).mean()

    # f) align to our uniform cycle grid
    cycle_df[sid] = daily.reindex(cycle_index)

# 4) Baseline‐subtract each mouse’s minimum, then grand‐mean
cycle_df_baselined = cycle_df.subtract(cycle_df.min())
grand_mean = cycle_df_baselined.mean(axis=1)
sem = cycle_df_baselined.sem(axis=1)

# 5) smooth both mean and SEM with a centered rolling window
window = 20
grand_mean_smooth = grand_mean.rolling(window=window, center=True, min_periods=1).mean()
sem_smooth = sem.rolling(window=window, center=True, min_periods=1).mean()

# 5) Plot mean ± SEM
plt.figure(figsize=(10,4))
x_hours = cycle_df_baselined.index.total_seconds() / 3600

plt.plot(x_hours, grand_mean_smooth, lw=2, label='Mean weight')
plt.fill_between(
    x_hours,
    grand_mean_smooth - sem_smooth,
    grand_mean_smooth + sem_smooth,
    alpha=0.3,
    label='± SEM'
)

plt.xlabel(f'Time since {light_off:02d}:00:00 (hours)')
plt.ylabel('Weight (baseline-subtracted)')
plt.title('Average 24 h weight cycle across all mice\n(Mean ± SEM)')
plt.xlim(0, 24)
plt.legend()
plt.tight_layout()
plt.show()

Baselined by subtracting the minimum of each subject's _smoothed_ 24h mean-day curve

In [ ]:
# 1) Flatten the “array-of-arrays” df into a long DataFrame
records = []
for _, row in social_weight_df_all_exps.iterrows():
    ts = pd.to_datetime(row['timestamps'])
    w = np.asarray(row['weight'], dtype=float)
    sids = row['subject_id']
    for t, weight, sid in zip(ts, w, sids):
        records.append({
            'timestamp': t,
            'subject_id': sid,
            'weight': weight
        })

df = pd.DataFrame.from_records(records)
df = df.sort_values(['subject_id', 'timestamp'])
# drop rows where subject_id is shorter than 11 characters, removes some erroneous entries
df = df[df['subject_id'].str.len() >= 11].reset_index(drop=True)

# 2) Parameters & 24 h cycle grid
sampling_freq = '10min'  # resample interval
# choose any date at 08:00 to define “day zero”
anchor = pd.Timestamp(f'2020-01-01 {light_off:02d}:00:00')

# build the 24 h cycle index
n_steps = int(pd.Timedelta('1D') / pd.Timedelta(sampling_freq))
cycle_index = pd.timedelta_range(start=0, periods=n_steps, freq=sampling_freq)

# will hold each mouse’s mean‐day (unsmoothed)
cycle_df = pd.DataFrame(index=cycle_index)

# 3) Loop over each mouse, resample, fold into 24 h, average days
for sid, grp in df.groupby('subject_id'):
    # a) series of weight vs time
    ser = grp.set_index('timestamp')['weight'].sort_index()
    # b) collapse any exact-duplicate timestamps
    ser = ser.groupby(level=0).mean()

    # c) resample into bins anchored at 08:00, then interpolate
    ser_rs = (
        ser
        .resample(sampling_freq, origin=anchor)
        .mean()
        .interpolate()
    )

    # d) convert each timestamp into its offset (mod 24 h) from the anchor
    offsets = (ser_rs.index - anchor) % pd.Timedelta('1D')
    ser_rs.index = offsets

    # e) average across all days for each offset
    daily = ser_rs.groupby(ser_rs.index).mean()

    # f) align to our uniform cycle grid
    cycle_df[sid] = daily.reindex(cycle_index)

# 4) Baseline‐subtract using each subject’s *smoothed* minimum
window = 20  # keep same smoothing window for per‐subject curves
# a) smooth each column (i.e., each subject’s 24 h curve)
cycle_df_smooth = cycle_df.rolling(window=window, center=True, min_periods=1).mean()

# b) find the minimum of each subject’s smoothed curve
minima_smooth = cycle_df_smooth.min()   # Series indexed by subject_id

# c) subtract that baseline from the UNSMOOTHED cycle for each subject
cycle_df_baselined = cycle_df.subtract(minima_smooth, axis=1)

# 5) Now compute grand‐mean and SEM (over baselined, unsmoothed curves)
grand_mean = cycle_df_baselined.mean(axis=1)
sem = cycle_df_baselined.sem(axis=1)

# 6) Smooth both grand‐mean and SEM for plotting
grand_mean_smooth = grand_mean.rolling(window=window, center=True, min_periods=1).mean()
sem_smooth = sem.rolling(window=window, center=True, min_periods=1).mean()

# 7) Plot mean ± SEM
plt.figure(figsize=(10, 4))
x_hours = cycle_df_baselined.index.total_seconds() / 3600

plt.plot(x_hours, grand_mean_smooth, lw=2, label='Mean weight (baselined)')
plt.fill_between(
    x_hours,
    grand_mean_smooth - sem_smooth,
    grand_mean_smooth + sem_smooth,
    alpha=0.3,
    label='± SEM'
)

plt.xlabel(f'Time since {light_off:02d}:00:00 (hours)')
plt.ylabel('Weight (baseline‐subtracted)')
plt.title('Average 24 h weight cycle across all mice\n(Mean ± SEM)')
plt.xlim(0, 24)
plt.legend()
plt.tight_layout()

svg_path = save_dir / "24h_weight_cycle.svg"
plt.savefig(str(svg_path), format='svg')

plt.show()